# 04e — Siamese CNN: Cosine Similarity / CosineEmbeddingLoss

## Obiettivo

Questo notebook studia la variante cosine della Siamese Network applicata
alle FCGR con `k=6`.

Il punto di partenza è la migliore configurazione ottenuta con
Euclidean Contrastive Loss:

- CNN encoder con GroupNorm;
- nessun Dropout;
- embedding dimension = 128;
- embedding L2-normalizzati;
- batch size = 128;
- learning rate iniziale = 5e-4;
- AdamW fused;
- AMP;
- forward dei due rami concatenata;
- pair dinamiche 50/50;
- split train/validation/test invariati e senza leakage;
- checkpoint selezionato tramite Validation ROC-AUC.

La modifica principale è:

Euclidean Contrastive Loss
→
Cosine similarity + CosineEmbeddingLoss

Il miglior margin euclideo precedente è:

margin = 1.25

Per embedding L2-normalizzati:

d² = 2 - 2 cos(θ)

quindi il valore geometricamente equivalente nel dominio cosine è:

cos_margin = 1 - 1.25² / 2 = 0.21875

Questo valore viene utilizzato come configurazione di riferimento,
ma NON viene assunto come margin cosine ottimale.

Verrà successivamente eseguita una cosine-margin ablation.

## Obiettivi secondari

Il notebook introduce inoltre ottimizzazioni della pipeline per ridurre
il tempo di training:

- meno trasferimenti GPU → CPU;
- trasferimenti CPU → GPU non bloccanti;
- DataLoader più leggero;
- forward concatenata;
- calcolo della cosine similarity senza operazioni duplicate;
- benchmark separato del DataLoader;
- eventuale torch.compile solo dopo benchmark.

## Metriche

Per ogni esperimento verranno analizzate:

- Validation ROC-AUC;
- Test ROC-AUC;
- Macro-F1;
- Balanced Accuracy;
- cosine similarity positiva;
- cosine similarity negativa;
- cosine gap;
- cosine d-prime;
- distanza euclidea positiva d+;
- distanza euclidea negativa d-;
- distance gap;
- euclidean d-prime.

In [1]:
from pathlib import Path

import json
import random
import time
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    get_worker_info,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Compute capability:",
        torch.cuda.get_device_capability(0),
    )

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(
        f"VRAM totale: {total_memory:.2f} GB"
    )

ENVIRONMENT
PyTorch: 2.12.0+cu126
CUDA disponibile: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Compute capability: (8, 6)
VRAM totale: 6.00 GB


In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_cosine"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# INPUT FILES
# ============================================================

PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)

MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)

VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)

TEST_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_test_pair_pool.tsv"
)


# ============================================================
# LOAD PAIR CONFIG
# ============================================================

with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    PAIR_CONFIG = json.load(file)


K = int(
    PAIR_CONFIG["k"]
)

TRAIN_PAIRS_PER_EPOCH = int(
    PAIR_CONFIG["train_pairs_per_epoch"]
)

VAL_PAIRS = int(
    PAIR_CONFIG["val_pairs"]
)

TEST_PAIRS = int(
    PAIR_CONFIG["test_pairs"]
)

POSITIVE_PAIR_PROBABILITY = float(
    PAIR_CONFIG["positive_probability"]
)

RANDOM_STATE = int(
    PAIR_CONFIG["random_state"]
)


# ============================================================
# FCGR CACHE
# ============================================================

FCGR_MEMMAP_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)

FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


print("=" * 60)
print("PROJECT")
print("=" * 60)

print("Project root:")
print(PROJECT_ROOT)

print()

print("Processed data:")
print(PROCESSED_DIR)

print()

print("Artifacts:")
print(ARTIFACTS_DIR)

print()

print("k:", K)
print(
    "Train pairs / epoch:",
    f"{TRAIN_PAIRS_PER_EPOCH:,}",
)
print(
    "Validation pairs:",
    f"{VAL_PAIRS:,}",
)
print(
    "Test pairs:",
    f"{TEST_PAIRS:,}",
)
print(
    "Positive probability:",
    POSITIVE_PAIR_PROBABILITY,
)
print(
    "Random state:",
    RANDOM_STATE,
)

PROJECT
Project root:
D:\Daria\Desktop\eccdna_fcgr_siamese

Processed data:
D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed

Artifacts:
D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_cosine

k: 6
Train pairs / epoch: 50,000
Validation pairs: 10,000
Test pairs: 10,000
Positive probability: 0.5
Random state: 42


In [3]:
# ============================================================
# MODEL
# ============================================================

EMBEDDING_DIM = 128


# ============================================================
# TRAINING
# ============================================================

BATCH_SIZE = 128

LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 50

EARLY_STOPPING_PATIENCE = 10

MIN_DELTA = 1e-4


# ============================================================
# COSINE LOSS
# ============================================================

EUCLIDEAN_REFERENCE_MARGIN = 1.25

REFERENCE_COSINE_MARGIN = (
    1.0
    - (EUCLIDEAN_REFERENCE_MARGIN ** 2) / 2.0
)


# Margin che testeremo DOPO la reference run.
COSINE_MARGIN_CANDIDATES = [
    0.00,
    0.10,
    REFERENCE_COSINE_MARGIN,
    0.35,
]


# ============================================================
# PERFORMANCE
# ============================================================

USE_AMP = True

PIN_MEMORY = torch.cuda.is_available()

NUM_WORKERS = 0

USE_TORCH_COMPILE = False


print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print("Embedding dim:", EMBEDDING_DIM)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Max epochs:", MAX_EPOCHS)
print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE,
)

print()

print(
    "Euclidean reference margin:",
    EUCLIDEAN_REFERENCE_MARGIN,
)

print(
    "Equivalent cosine margin:",
    REFERENCE_COSINE_MARGIN,
)

print(
    "Cosine margin candidates:",
    COSINE_MARGIN_CANDIDATES,
)

print()

print("AMP:", USE_AMP)
print("Pin memory:", PIN_MEMORY)
print("Num workers:", NUM_WORKERS)
print("torch.compile:", USE_TORCH_COMPILE)

EXPERIMENT CONFIGURATION
Embedding dim: 128
Batch size: 128
Learning rate: 0.0005
Weight decay: 0.0001
Max epochs: 50
Early stopping patience: 10

Euclidean reference margin: 1.25
Equivalent cosine margin: 0.21875
Cosine margin candidates: [0.0, 0.1, 0.21875, 0.35]

AMP: True
Pin memory: True
Num workers: 0
torch.compile: False


In [4]:
def set_seed(seed: int) -> None:

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# CUDA PERFORMANCE
# ============================================================

if DEVICE.type == "cuda":

    # Input FCGR sempre 64x64:
    # cuDNN può scegliere il kernel convoluzionale più efficiente.
    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print("=" * 60)
print("DEVICE CONFIGURATION")
print("=" * 60)

print("Device:", DEVICE)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark,
)

if DEVICE.type == "cuda":

    print(
        "Allocated VRAM:",
        f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB",
    )

    print(
        "Reserved VRAM:",
        f"{torch.cuda.memory_reserved() / 1024**2:.1f} MB",
    )

DEVICE CONFIGURATION
Device: cuda
cuDNN benchmark: True
Allocated VRAM: 0.0 MB
Reserved VRAM: 0.0 MB


In [5]:
# ============================================================
# CELL 5 — DATA / PATH DIAGNOSTICS + LOAD
# ============================================================

print("=" * 70)
print("PATH DIAGNOSTICS")
print("=" * 70)

print("Current working directory:")
print(Path.cwd().resolve())

print()

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print()

print("PROCESSED_DIR:")
print(PROCESSED_DIR)

print()


# ============================================================
# HELPER: RESOLVE FILE IF EXPECTED PATH IS WRONG
# ============================================================

def resolve_file(
    expected_path,
    filename,
    search_root=None,
):
    """
    Usa expected_path se esiste.
    Altrimenti cerca filename ricorsivamente dentro search_root.
    """

    expected_path = Path(expected_path)

    if expected_path.exists():

        print(f"[OK] {filename}")
        print(f"     {expected_path}")

        return expected_path


    print(f"[MISSING] {expected_path}")


    if search_root is None:
        search_root = PROJECT_ROOT


    matches = list(
        Path(search_root).rglob(filename)
    )


    if len(matches) == 0:

        raise FileNotFoundError(
            f"\nFile non trovato: {filename}\n"
            f"Path previsto:\n{expected_path}\n\n"
            f"Nessuna copia trovata dentro:\n"
            f"{search_root}"
        )


    if len(matches) > 1:

        print()
        print(
            f"ATTENZIONE: trovate {len(matches)} copie "
            f"di {filename}:"
        )

        for match in matches:
            print("  ", match)

        print()
        print(
            "Uso la prima copia trovata:"
        )


    resolved = matches[0]

    print(f"[FOUND] {filename}")
    print(f"        {resolved}")

    return resolved


# ============================================================
# RESOLVE REQUIRED FILES
# ============================================================

PAIR_CONFIG_PATH = resolve_file(
    PAIR_CONFIG_PATH,
    "siamese_pair_config.json",
)


MANIFEST_PATH = resolve_file(
    MANIFEST_PATH,
    "siamese_primary_manifest.tsv",
)


VAL_POOL_PATH = resolve_file(
    VAL_POOL_PATH,
    "siamese_val_pair_pool.tsv",
)


TEST_POOL_PATH = resolve_file(
    TEST_POOL_PATH,
    "siamese_test_pair_pool.tsv",
)


FCGR_MEMMAP_PATH = resolve_file(
    FCGR_MEMMAP_PATH,
    f"fcgr_k{K}.npy",
)


FCGR_INDEX_PATH = resolve_file(
    FCGR_INDEX_PATH,
    f"fcgr_k{K}_index.tsv",
)


# ============================================================
# LOAD FILES
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
)


val_pool = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
)


test_pool = pd.read_csv(
    TEST_POOL_PATH,
    sep="\t",
)


# ============================================================
# PRINT RAW STRUCTURE
# ============================================================

print()
print("=" * 70)
print("MANIFEST STRUCTURE")
print("=" * 70)

print("Shape:")
print(manifest.shape)

print()

print("Columns:")
print(
    manifest.columns.tolist()
)


print()
print("=" * 70)
print("FCGR INDEX STRUCTURE")
print("=" * 70)

print("Shape:")
print(fcgr_index.shape)

print()

print("Columns:")
print(
    fcgr_index.columns.tolist()
)


print()
print("=" * 70)
print("VALIDATION PAIR POOL STRUCTURE")
print("=" * 70)

print("Shape:")
print(val_pool.shape)

print()

print("Columns:")
print(
    val_pool.columns.tolist()
)


print()
print("=" * 70)
print("TEST PAIR POOL STRUCTURE")
print("=" * 70)

print("Shape:")
print(test_pool.shape)

print()

print("Columns:")
print(
    test_pool.columns.tolist()
)


# ============================================================
# DATASET SEMANTIC COLUMNS
# ============================================================

ID_COL = "id"

CLASS_COL = "class_id"

SPLIT_COL = "split_cluster"


required_columns = {
    ID_COL,
    CLASS_COL,
    SPLIT_COL,
}


missing_columns = (
    required_columns
    -
    set(manifest.columns)
)


if missing_columns:

    raise KeyError(
        "Colonne richieste mancanti nel manifest: "
        f"{sorted(missing_columns)}"
    )


# ============================================================
# CHECK DISEASE_GROUP / CLASS_ID RELATION
# ============================================================

if "class_id" in manifest.columns:

    group_to_id = (
        manifest
        .groupby("disease_group")["class_id"]
        .nunique()
    )

    id_to_group = (
        manifest
        .groupby("class_id")["disease_group"]
        .nunique()
    )


    class_mapping_is_one_to_one = (
        group_to_id.max() == 1
        and
        id_to_group.max() == 1
    )


    print()
    print("=" * 70)
    print("CLASS MAPPING CHECK")
    print("=" * 70)

    print(
        "disease_group <-> class_id one-to-one:",
        class_mapping_is_one_to_one,
    )

    print()

    display(
        manifest[
            [
                "disease_group",
                "class_id",
            ]
        ]
        .drop_duplicates()
        .sort_values("class_id")
        .reset_index(drop=True)
    )


# ============================================================
# CHECK SPLIT VALUES
# ============================================================

print()
print("=" * 70)
print("RAW split_cluster VALUES")
print("=" * 70)

print(
    manifest[SPLIT_COL]
    .value_counts(
        dropna=False
    )
)


print()
print("Unique values:")

print(
    manifest[SPLIT_COL]
    .drop_duplicates()
    .tolist()
)


# ============================================================
# NORMALIZE SPLIT
# ============================================================

def normalize_split(value):

    value = (
        str(value)
        .strip()
        .lower()
    )

    aliases = {

        "training": "train",
        "tr": "train",

        "validation": "val",
        "valid": "val",
        "dev": "val",

        "testing": "test",
        "te": "test",
    }

    return aliases.get(
        value,
        value,
    )


manifest["_split_normalized"] = (
    manifest[SPLIT_COL]
    .map(normalize_split)
)


print()
print("=" * 70)
print("NORMALIZED SPLIT VALUES")
print("=" * 70)

print(
    manifest["_split_normalized"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# LOAD FCGR MEMMAP
# ============================================================

fcgr_memmap_check = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r",
)


print()
print("=" * 70)
print("FCGR CACHE")
print("=" * 70)

print(
    "Path:",
    FCGR_MEMMAP_PATH,
)

print(
    "Shape:",
    fcgr_memmap_check.shape,
)

print(
    "dtype:",
    fcgr_memmap_check.dtype,
)

print(
    "Index rows:",
    len(fcgr_index),
)


# ============================================================
# FINAL DIAGNOSTIC
# ============================================================

print()
print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

print(
    "ID_COL:",
    ID_COL,
)

print(
    "CLASS_COL:",
    CLASS_COL,
)

print(
    "SPLIT_COL:",
    SPLIT_COL,
)

PATH DIAGNOSTICS
Current working directory:
D:\Daria\Desktop\eccdna_fcgr_siamese\notebooks

PROJECT_ROOT:
D:\Daria\Desktop\eccdna_fcgr_siamese

PROCESSED_DIR:
D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed

[OK] siamese_pair_config.json
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_pair_config.json
[OK] siamese_primary_manifest.tsv
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_primary_manifest.tsv
[OK] siamese_val_pair_pool.tsv
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_val_pair_pool.tsv
[OK] siamese_test_pair_pool.tsv
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_test_pair_pool.tsv
[OK] fcgr_k6.npy
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\fcgr_cache\fcgr_k6.npy
[OK] fcgr_k6_index.tsv
     D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\fcgr_cache\fcgr_k6_index.tsv

MANIFEST STRUCTURE
Shape:
(665681, 12)

Columns:
['id', 'disease', 'disease_clean', 'disease_group', 'class_id', 'split

,disease_group,class_id
0,cancer,0
1,healthy,1
2,cancer,2
3,cancer,3
4,cancer,4
5,cancer,5
6,non_cancer_disease,6
7,cancer,7
8,cancer,8
9,non_cancer_disease,9



RAW split_cluster VALUES
split_cluster
val      358124
test     181292
train    126265
Name: count, dtype: int64

Unique values:
['train', 'val', 'test']

NORMALIZED SPLIT VALUES
_split_normalized
val      358124
test     181292
train    126265
Name: count, dtype: int64

FCGR CACHE
Path: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\fcgr_cache\fcgr_k6.npy
Shape: (150272, 64, 64)
dtype: float32
Index rows: 150272

DIAGNOSTIC COMPLETE
ID_COL: id
CLASS_COL: class_id
SPLIT_COL: split_cluster


In [6]:
# ============================================================
# CELL 6 — PREPARE DATA POOLS
# Baseline-compatible
# ============================================================

metadata = (
    manifest
    .copy()
    .reset_index(drop=True)
)


val_pair_pool = (
    val_pool
    .copy()
    .reset_index(drop=True)
)


test_pair_pool = (
    test_pool
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# NORMALIZE IDS
# ============================================================

for dataframe in [
    metadata,
    val_pair_pool,
    test_pair_pool,
]:

    dataframe["id"] = (
        dataframe["id"]
        .astype(str)
    )


# ============================================================
# TRAIN SPLIT
# ============================================================

train_metadata = (
    metadata[
        metadata["split_cluster"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# CLASS ID
# ============================================================

for dataframe in [
    train_metadata,
    val_pair_pool,
    test_pair_pool,
]:

    if "class_id" not in dataframe.columns:

        raise KeyError(
            "Manca la colonna 'class_id'.\n"
            f"Colonne disponibili: "
            f"{dataframe.columns.tolist()}"
        )

    dataframe["class_id"] = (
        dataframe["class_id"]
        .astype(int)
    )


# ============================================================
# VALIDATION
# ============================================================

if len(train_metadata) == 0:

    raise RuntimeError(
        "Training split vuoto."
    )


for name, dataframe in [

    ("train", train_metadata),

    ("validation", val_pair_pool),

    ("test", test_pair_pool),

]:

    if dataframe["id"].duplicated().any():

        duplicated = int(
            dataframe["id"]
            .duplicated()
            .sum()
        )

        print(
            f"ATTENZIONE: {name} contiene "
            f"{duplicated} ID duplicati."
        )


# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("DATA POOLS")
print("=" * 70)

print(
    "Train samples:",
    f"{len(train_metadata):,}"
)

print(
    "Validation pool:",
    f"{len(val_pair_pool):,}"
)

print(
    "Test pool:",
    f"{len(test_pair_pool):,}"
)

print()

print(
    "Train classes:",
    train_metadata[
        "class_id"
    ].nunique()
)

print(
    "Validation classes:",
    val_pair_pool[
        "class_id"
    ].nunique()
)

print(
    "Test classes:",
    test_pair_pool[
        "class_id"
    ].nunique()
)

print()

print("=" * 70)
print("TRAIN CLASS DISTRIBUTION")
print("=" * 70)

display(
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_samples")
    .to_frame()
)

DATA POOLS
Train samples: 126,265
Validation pool: 12,937
Test pool: 11,070

Train classes: 18
Validation classes: 18
Test classes: 18

TRAIN CLASS DISTRIBUTION


,n_samples
class_id,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,10000


In [7]:
# ============================================================
# CELL 7 — LOAD FCGR MEMMAP + INDEX
# Same mapping used in previous Siamese notebooks
# ============================================================

fcgr_memmap = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


# ============================================================
# EXPECTED FCGR INDEX FORMAT
# ============================================================

required_fcgr_columns = {
    "id",
    "fcgr_row",
}


missing_fcgr_columns = (
    required_fcgr_columns
    -
    set(fcgr_index.columns)
)


if missing_fcgr_columns:

    raise KeyError(
        "Il FCGR index non ha la struttura "
        "usata nei notebook precedenti.\n"
        f"Colonne mancanti: "
        f"{sorted(missing_fcgr_columns)}\n"
        f"Colonne presenti: "
        f"{fcgr_index.columns.tolist()}"
    )


# ============================================================
# NORMALIZE
# ============================================================

fcgr_index["id"] = (
    fcgr_index["id"]
    .astype(str)
)


fcgr_index["fcgr_row"] = (
    fcgr_index["fcgr_row"]
    .astype(np.int64)
)


# ============================================================
# ID -> MEMMAP ROW
# ============================================================

id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"],
    )
)


# ============================================================
# CHECKS
# ============================================================

if fcgr_index["id"].duplicated().any():

    raise RuntimeError(
        "Sono presenti ID duplicati "
        "nel FCGR index."
    )


if fcgr_index["fcgr_row"].duplicated().any():

    raise RuntimeError(
        "Sono presenti fcgr_row duplicate "
        "nel FCGR index."
    )


if (
    fcgr_index["fcgr_row"].min() < 0
    or
    fcgr_index["fcgr_row"].max()
    >= fcgr_memmap.shape[0]
):

    raise RuntimeError(
        "fcgr_row contiene indici "
        "fuori dai limiti del memmap."
    )


# ============================================================
# CHECK ALL DATASET IDS
# ============================================================

all_required_ids = set(
    pd.concat(
        [
            train_metadata["id"],
            val_pair_pool["id"],
            test_pair_pool["id"],
        ],
        ignore_index=True,
    ).astype(str)
)


missing_ids = (
    all_required_ids
    -
    set(id_to_fcgr_row.keys())
)


if missing_ids:

    raise RuntimeError(
        f"{len(missing_ids)} sample non sono "
        "presenti nella cache FCGR.\n"
        f"Primi ID mancanti: "
        f"{list(missing_ids)[:10]}"
    )


# ============================================================
# BASELINE-COMPATIBILITY CHECKS
# ============================================================

assert (
    fcgr_memmap.dtype
    ==
    np.float32
)


assert (
    len(id_to_fcgr_row)
    ==
    fcgr_memmap.shape[0]
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("FCGR CACHE")
print("=" * 70)

print(
    "FCGR shape:",
    fcgr_memmap.shape
)

print(
    "FCGR dtype:",
    fcgr_memmap.dtype
)

print(
    "Index rows:",
    len(fcgr_index)
)

print(
    "Mapped IDs:",
    len(id_to_fcgr_row)
)

print()

print(
    "fcgr_row min:",
    fcgr_index["fcgr_row"].min()
)

print(
    "fcgr_row max:",
    fcgr_index["fcgr_row"].max()
)

print()

print(
    "All train/val/test IDs found:",
    "YES"
)

FCGR CACHE
FCGR shape: (150272, 64, 64)
FCGR dtype: float32
Index rows: 150272
Mapped IDs: 150272

fcgr_row min: 0
fcgr_row max: 150271

All train/val/test IDs found: YES


In [8]:
# ============================================================
# CELL 8 — SIAMESE PAIR DATASET
# ============================================================

class SiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_fcgr_row,
        pairs_per_epoch,
        positive_probability=0.5,
        seed=42,
        deterministic=False,
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.id_to_fcgr_row = (
            id_to_fcgr_row
        )

        self.pairs_per_epoch = int(
            pairs_per_epoch
        )

        self.positive_probability = float(
            positive_probability
        )

        self.seed = int(seed)

        self.deterministic = bool(
            deterministic
        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        self.metadata["id"] = (
            self.metadata["id"]
            .astype(str)
        )

        self.metadata["class_id"] = (
            self.metadata["class_id"]
            .astype(int)
        )


        # ----------------------------------------------------
        # CLASS -> FCGR ROWS
        #
        # Precomputed once, rather than using pandas
        # inside __getitem__.
        # ----------------------------------------------------

        self.class_to_rows = {}

        for class_id, group in (
            self.metadata.groupby(
                "class_id",
                sort=False,
            )
        ):

            rows = np.array(
                [
                    self.id_to_fcgr_row[
                        sample_id
                    ]

                    for sample_id
                    in group["id"]
                ],
                dtype=np.int64,
            )

            self.class_to_rows[
                int(class_id)
            ] = rows


        self.classes = np.array(
            list(
                self.class_to_rows.keys()
            ),
            dtype=np.int64,
        )


        # Classi utilizzabili per positive pair:
        # servono almeno 2 sample.

        self.positive_classes = np.array(
            [
                class_id

                for class_id, rows
                in self.class_to_rows.items()

                if len(rows) >= 2
            ],
            dtype=np.int64,
        )


        if len(self.classes) < 2:

            raise RuntimeError(
                "Servono almeno 2 classi "
                "per generare negative pairs."
            )


        if len(self.positive_classes) == 0:

            raise RuntimeError(
                "Nessuna classe contiene almeno "
                "2 sample per positive pairs."
            )


        # Worker-local RNG.
        self._rng = None
        self._rng_seed = None


    def __len__(self):

        return self.pairs_per_epoch


    # ========================================================
    # RNG
    # ========================================================

    def _get_rng(
        self,
        index,
    ):

        # ----------------------------------------------------
        # Validation / test:
        # same index => same pair.
        # ----------------------------------------------------

        if self.deterministic:

            return np.random.default_rng(
                self.seed + int(index)
            )


        # ----------------------------------------------------
        # Training:
        # worker-specific random generator.
        # ----------------------------------------------------

        worker_info = get_worker_info()


        if worker_info is None:

            worker_seed = self.seed

        else:

            worker_seed = int(
                worker_info.seed
            )


        if (
            self._rng is None
            or
            self._rng_seed != worker_seed
        ):

            self._rng = (
                np.random.default_rng(
                    worker_seed
                )
            )

            self._rng_seed = (
                worker_seed
            )


        return self._rng


    # ========================================================
    # FCGR
    # ========================================================

    def _load_fcgr(
        self,
        row,
    ):

        # copy=True:
        # evita warning PyTorch su array memmap read-only.

        fcgr = np.array(
            self.fcgr_memmap[
                int(row)
            ],
            dtype=np.float32,
            copy=True,
        )


        # Cache possibilities:
        #
        # (64, 64)
        # oppure
        # (1, 64, 64)

        if fcgr.ndim == 2:

            fcgr = fcgr[
                None,
                :,
                :
            ]


        elif (
            fcgr.ndim == 3
            and
            fcgr.shape[0] == 1
        ):

            pass


        else:

            raise RuntimeError(
                "Shape FCGR non supportata: "
                f"{fcgr.shape}"
            )


        return torch.from_numpy(
            fcgr
        )


    # ========================================================
    # GETITEM
    # ========================================================

    def __getitem__(
        self,
        index,
    ):

        rng = self._get_rng(
            index
        )


        # ----------------------------------------------------
        # Decide positive / negative
        # ----------------------------------------------------

        positive_pair = (
            rng.random()
            <
            self.positive_probability
        )


        # ====================================================
        # POSITIVE PAIR
        # ====================================================

        if positive_pair:

            class_id = int(
                rng.choice(
                    self.positive_classes
                )
            )


            rows = (
                self.class_to_rows[
                    class_id
                ]
            )


            row1, row2 = rng.choice(
                rows,
                size=2,
                replace=False,
            )


            target = 1.0


        # ====================================================
        # NEGATIVE PAIR
        # ====================================================

        else:

            class1, class2 = rng.choice(
                self.classes,
                size=2,
                replace=False,
            )


            row1 = rng.choice(
                self.class_to_rows[
                    int(class1)
                ]
            )


            row2 = rng.choice(
                self.class_to_rows[
                    int(class2)
                ]
            )


            target = 0.0


        # ====================================================
        # FCGR
        # ====================================================

        x1 = self._load_fcgr(
            row1
        )

        x2 = self._load_fcgr(
            row2
        )


        # Stack once:
        #
        # [2, 1, 64, 64]

        pair = torch.stack(
            [
                x1,
                x2,
            ],
            dim=0,
        )


        return (
            pair,
            torch.tensor(
                target,
                dtype=torch.float32,
            ),
        )

In [9]:
# ============================================================
# CELL 9 — CREATE DATASETS
# ============================================================

train_dataset = SiamesePairDataset(

    metadata=train_metadata,

    fcgr_memmap=fcgr_memmap,

    id_to_fcgr_row=id_to_fcgr_row,

    pairs_per_epoch=(
        TRAIN_PAIRS_PER_EPOCH
    ),

    positive_probability=(
        POSITIVE_PAIR_PROBABILITY
    ),

    seed=RANDOM_STATE,

    deterministic=False,
)


val_dataset = SiamesePairDataset(

    metadata=val_pair_pool,

    fcgr_memmap=fcgr_memmap,

    id_to_fcgr_row=id_to_fcgr_row,

    pairs_per_epoch=(
        VAL_PAIRS
    ),

    positive_probability=(
        POSITIVE_PAIR_PROBABILITY
    ),

    seed=(
        RANDOM_STATE + 10_000
    ),

    deterministic=True,
)


test_dataset = SiamesePairDataset(

    metadata=test_pair_pool,

    fcgr_memmap=fcgr_memmap,

    id_to_fcgr_row=id_to_fcgr_row,

    pairs_per_epoch=(
        TEST_PAIRS
    ),

    positive_probability=(
        POSITIVE_PAIR_PROBABILITY
    ),

    seed=(
        RANDOM_STATE + 20_000
    ),

    deterministic=True,
)


print("=" * 70)
print("DATASETS")
print("=" * 70)

print(
    "Train pairs / epoch:",
    f"{len(train_dataset):,}",
)

print(
    "Validation pairs:",
    f"{len(val_dataset):,}",
)

print(
    "Test pairs:",
    f"{len(test_dataset):,}",
)

DATASETS
Train pairs / epoch: 50,000
Validation pairs: 10,000
Test pairs: 10,000


In [10]:
# ============================================================
# CELL 10 — DATASET SANITY CHECK
# ============================================================

pair, target = (
    train_dataset[0]
)


print("=" * 70)
print("SINGLE PAIR")
print("=" * 70)

print(
    "Pair shape:",
    pair.shape,
)

print(
    "Pair dtype:",
    pair.dtype,
)

print(
    "Target:",
    target.item(),
)

print()

print(
    "FCGR 1 min/max:",
    float(pair[0].min()),
    float(pair[0].max()),
)

print(
    "FCGR 2 min/max:",
    float(pair[1].min()),
    float(pair[1].max()),
)


assert (
    pair.ndim == 4
)

assert (
    pair.shape[0] == 2
)

assert (
    pair.shape[1] == 1
)

assert (
    pair.dtype
    ==
    torch.float32
)

assert (
    target.item()
    in
    {0.0, 1.0}
)


print()
print("Single pair: OK")


# ============================================================
# CHECK POSITIVE / NEGATIVE RATIO
# ============================================================

N_CHECK = 2000


targets = np.array(
    [
        float(
            train_dataset[i][1]
        )

        for i in range(
            N_CHECK
        )
    ]
)


print()
print("=" * 70)
print("TRAIN PAIR DISTRIBUTION")
print("=" * 70)

print(
    "Checked:",
    f"{N_CHECK:,}",
)

print(
    "Positive:",
    int(targets.sum()),
)

print(
    "Negative:",
    int(
        N_CHECK
        -
        targets.sum()
    ),
)

print(
    "Positive ratio:",
    f"{targets.mean():.4f}",
)

SINGLE PAIR
Pair shape: torch.Size([2, 1, 64, 64])
Pair dtype: torch.float32
Target: 0.0

FCGR 1 min/max: 0.0 0.007792207878082991
FCGR 2 min/max: 0.0 0.006956521887332201

Single pair: OK

TRAIN PAIR DISTRIBUTION
Checked: 2,000
Positive: 1010
Negative: 990
Positive ratio: 0.5050


In [11]:
# ============================================================
# CELL 11 — DETERMINISTIC VAL / TEST CHECK
# ============================================================

val_pair_a, val_target_a = (
    val_dataset[123]
)

val_pair_b, val_target_b = (
    val_dataset[123]
)


same_pair = torch.equal(
    val_pair_a,
    val_pair_b,
)

same_target = torch.equal(
    val_target_a,
    val_target_b,
)


print("=" * 70)
print("DETERMINISTIC VALIDATION CHECK")
print("=" * 70)

print(
    "Same FCGR pair:",
    same_pair,
)

print(
    "Same target:",
    same_target,
)


assert same_pair

assert same_target


print()
print(
    "Validation deterministic sampling: OK"
)

DETERMINISTIC VALIDATION CHECK
Same FCGR pair: True
Same target: True

Validation deterministic sampling: OK


In [12]:
# ============================================================
# CELL 12 — DATALOADER FACTORY
# ============================================================

def create_loader(
    dataset,
    num_workers,
    batch_size=BATCH_SIZE,
):

    kwargs = {

        "dataset":
            dataset,

        "batch_size":
            batch_size,

        # Pair generation is already random for train.
        "shuffle":
            False,

        "num_workers":
            int(num_workers),

        "pin_memory":
            PIN_MEMORY,

        "drop_last":
            False,
    }


    if num_workers > 0:

        kwargs.update(
            {
                "persistent_workers":
                    True,

                "prefetch_factor":
                    2,
            }
        )


    return DataLoader(
        **kwargs
    )

In [13]:
# ============================================================
# CELL 13 — FINAL DATALOADER CONFIGURATION
# ============================================================

# Windows + Jupyter:
# multiprocessing workers can hang when Dataset classes
# are defined directly inside the notebook.
#
# num_workers=0 already provides ~7.7k pairs/s,
# therefore data loading is not currently a bottleneck.

NUM_WORKERS = 0


print("=" * 70)
print("FINAL DATALOADER CONFIGURATION")
print("=" * 70)

print(
    "NUM_WORKERS:",
    NUM_WORKERS
)

print(
    "PIN_MEMORY:",
    PIN_MEMORY
)

print(
    "Reason:",
    "num_workers=0 already reaches ~7.7k pairs/s"
)

print(
    "Multiprocessing:",
    "disabled for notebook stability"
)

FINAL DATALOADER CONFIGURATION
NUM_WORKERS: 0
PIN_MEMORY: True
Reason: num_workers=0 already reaches ~7.7k pairs/s
Multiprocessing: disabled for notebook stability


In [14]:
# ============================================================
# CELL 14 — FINAL DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,
)


val_loader = DataLoader(
    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,
)


test_loader = DataLoader(
    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,
)


print("=" * 70)
print("FINAL DATALOADERS")
print("=" * 70)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

print()

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Num workers:",
    NUM_WORKERS
)

print(
    "Pin memory:",
    PIN_MEMORY
)

FINAL DATALOADERS
Train batches: 391
Validation batches: 79
Test batches: 79

Batch size: 128
Num workers: 0
Pin memory: True


In [15]:
# ============================================================
# CELL 15 — BATCH SANITY CHECK
# ============================================================

pair_batch, target_batch = next(
    iter(train_loader)
)


print("=" * 70)
print("BATCH SANITY CHECK")
print("=" * 70)

print(
    "Pair batch shape:",
    pair_batch.shape
)

print(
    "Target batch shape:",
    target_batch.shape
)

print()

print(
    "Pair dtype:",
    pair_batch.dtype
)

print(
    "Target dtype:",
    target_batch.dtype
)

print()

print(
    "Positive pairs:",
    int(target_batch.sum())
)

print(
    "Negative pairs:",
    int(
        len(target_batch)
        -
        target_batch.sum()
    )
)


assert pair_batch.ndim == 5

assert pair_batch.shape[1] == 2

assert pair_batch.shape[2] == 1

assert pair_batch.shape[-2:] == (
    64,
    64,
)

assert target_batch.ndim == 1


print()
print("Batch format: OK")

BATCH SANITY CHECK
Pair batch shape: torch.Size([128, 2, 1, 64, 64])
Target batch shape: torch.Size([128])

Pair dtype: torch.float32
Target dtype: torch.float32

Positive pairs: 63
Negative pairs: 65

Batch format: OK


In [16]:
# ============================================================
# CELL 16 — FCGR CNN ENCODER
# Same architecture as Baseline C
# ============================================================

class FCGRCNNEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        self.features = nn.Sequential(

            # ==================================================
            # 64 x 64
            # ==================================================

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=32,
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # ==================================================
            # 32 x 32
            # ==================================================

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=64,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # ==================================================
            # 16 x 16
            # ==================================================

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=128,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # ==================================================
            # 8 x 8
            # ==================================================

            nn.Conv2d(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=128,
            ),

            nn.ReLU(
                inplace=True
            ),


            # ==================================================
            # 4 x 4
            # ==================================================

            nn.AdaptiveAvgPool2d(
                (4, 4)
            ),
        )


        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim,
            ),
        )


    def forward(
        self,
        x,
    ):

        x = self.features(
            x
        )

        z = self.projection(
            x
        )


        # ------------------------------------------------------
        # Fundamental for cosine metric learning
        # ------------------------------------------------------

        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8,
        )


        return z

In [17]:
# ============================================================
# CELL 17 — SIAMESE NETWORK
# Optimized concatenated forward
# ============================================================

class SiameseNetwork(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()

        self.encoder = FCGRCNNEncoder(
            embedding_dim=embedding_dim
        )


    def forward(
        self,
        pair_batch,
    ):

        # pair_batch:
        #
        # [B, 2, 1, 64, 64]

        batch_size = (
            pair_batch.shape[0]
        )


        # ------------------------------------------------------
        # Merge Siamese branches
        #
        # [B, 2, 1, 64, 64]
        #
        # ->
        #
        # [2B, 1, 64, 64]
        # ------------------------------------------------------

        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        # Single encoder forward

        z = self.encoder(
            x
        )


        # ------------------------------------------------------
        # Restore pair dimension
        #
        # [2B, D]
        #
        # ->
        #
        # [B, 2, D]
        # ------------------------------------------------------

        z = z.reshape(
            batch_size,
            2,
            -1,
        )


        z1 = z[:, 0, :]

        z2 = z[:, 1, :]


        return z1, z2

In [18]:
# ============================================================
# CELL 18 — COSINE SIMILARITY
# Official PyTorch implementation
# ============================================================

def cosine_similarity(
    z1,
    z2,
):

    return F.cosine_similarity(
        z1,
        z2,
        dim=1,
        eps=1e-8,
    )

In [19]:
# ============================================================
# CELL 19 — COSINE EMBEDDING LOSS
# Official PyTorch implementation
# ============================================================

class CosineMetricLoss(nn.Module):

    def __init__(
        self,
        margin,
    ):

        super().__init__()

        self.margin = float(
            margin
        )


        self.loss_fn = (
            nn.CosineEmbeddingLoss(

                margin=self.margin,

                reduction="mean",
            )
        )


    def forward(
        self,
        z1,
        z2,
        target,
    ):

        # Dataset convention:
        #
        # 1 = positive pair
        # 0 = negative pair
        #
        # CosineEmbeddingLoss convention:
        #
        # +1 = positive
        # -1 = negative

        cosine_target = torch.where(

            target > 0.5,

            torch.ones_like(
                target
            ),

            -torch.ones_like(
                target
            ),
        )


        loss = self.loss_fn(

            z1,

            z2,

            cosine_target,
        )


        similarity = (
            F.cosine_similarity(

                z1,

                z2,

                dim=1,

                eps=1e-8,
            )
        )


        return (
            loss,
            similarity,
        )

In [20]:
# ============================================================
# CELL 20 — INITIAL EMBEDDING DIAGNOSTIC
#
# Goal:
# Check whether the cosine embedding space is already
# collapsed BEFORE any training step.
# ============================================================

set_seed(
    RANDOM_STATE
)


# ============================================================
# FRESH UNTRAINED MODEL
# ============================================================

diagnostic_model = SiameseNetwork(
    embedding_dim=EMBEDDING_DIM
).to(
    DEVICE
)


diagnostic_model.eval()


# ============================================================
# AMP
# Defined locally so this cell does not depend on later cells.
# ============================================================

diagnostic_amp_enabled = (
    USE_AMP
    and
    DEVICE.type == "cuda"
)


# ============================================================
# STORAGE
# ============================================================

all_similarities = []

all_targets = []


MAX_DIAGNOSTIC_BATCHES = 30


# ============================================================
# FORWARD
# ============================================================

with torch.inference_mode():

    for batch_index, (
        pair_batch,
        target_batch,
    ) in enumerate(
        val_loader
    ):

        if (
            batch_index
            >=
            MAX_DIAGNOSTIC_BATCHES
        ):
            break


        # ----------------------------------------------------
        # CPU -> GPU
        # ----------------------------------------------------

        pair_batch = pair_batch.to(
            DEVICE,
            non_blocking=True,
        )


        # ----------------------------------------------------
        # Encoder forward
        # ----------------------------------------------------

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=diagnostic_amp_enabled,
        ):

            z1, z2 = diagnostic_model(
                pair_batch
            )


        # ----------------------------------------------------
        # Compute cosine explicitly in FP32
        # ----------------------------------------------------

        similarity = F.cosine_similarity(
            z1.float(),
            z2.float(),
            dim=1,
            eps=1e-8,
        )


        all_similarities.append(
            similarity.cpu()
        )


        all_targets.append(
            target_batch.cpu()
        )


# ============================================================
# CONCATENATE
# ============================================================

initial_similarities = (
    torch.cat(
        all_similarities
    )
    .numpy()
)


initial_targets = (
    torch.cat(
        all_targets
    )
    .numpy()
    .astype(np.int64)
)


# ============================================================
# POSITIVE / NEGATIVE DISTRIBUTIONS
# ============================================================

positive_initial = (
    initial_similarities[
        initial_targets == 1
    ]
)


negative_initial = (
    initial_similarities[
        initial_targets == 0
    ]
)


if (
    len(positive_initial) == 0
    or
    len(negative_initial) == 0
):

    raise RuntimeError(
        "La diagnostica non contiene entrambe "
        "le classi positive/negative."
    )


# ============================================================
# ROC-AUC BEFORE TRAINING
# ============================================================

initial_auc = roc_auc_score(
    initial_targets,
    initial_similarities,
)


# ============================================================
# OUTPUT
# ============================================================

print("=" * 70)
print("UNTRAINED EMBEDDING DIAGNOSTIC")
print("=" * 70)

print(
    "Samples checked:",
    f"{len(initial_targets):,}"
)

print(
    "Positive pairs:",
    f"{len(positive_initial):,}"
)

print(
    "Negative pairs:",
    f"{len(negative_initial):,}"
)

print()


print(
    "All cosine mean:",
    f"{initial_similarities.mean():.6f}"
)

print(
    "All cosine std:",
    f"{initial_similarities.std():.6f}"
)

print()


print(
    "Positive cosine mean:",
    f"{positive_initial.mean():.6f}"
)

print(
    "Negative cosine mean:",
    f"{negative_initial.mean():.6f}"
)


initial_gap = (
    positive_initial.mean()
    -
    negative_initial.mean()
)


print(
    "Initial cosine gap:",
    f"{initial_gap:.6f}"
)

print()


print(
    "Initial ROC-AUC:",
    f"{initial_auc:.6f}"
)

print()


print(
    "Min:",
    f"{initial_similarities.min():.6f}"
)

print(
    "25%:",
    f"{np.quantile(initial_similarities, 0.25):.6f}"
)

print(
    "Median:",
    f"{np.median(initial_similarities):.6f}"
)

print(
    "75%:",
    f"{np.quantile(initial_similarities, 0.75):.6f}"
)

print(
    "Max:",
    f"{initial_similarities.max():.6f}"
)


# ============================================================
# CLEANUP
# ============================================================

del diagnostic_model

gc.collect()


if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

UNTRAINED EMBEDDING DIAGNOSTIC
Samples checked: 3,840
Positive pairs: 1,985
Negative pairs: 1,855

All cosine mean: 0.999979
All cosine std: 0.000077

Positive cosine mean: 0.999979
Negative cosine mean: 0.999979
Initial cosine gap: -0.000000

Initial ROC-AUC: 0.544810

Min: 0.998193
25%: 0.999986
Median: 0.999993
75%: 0.999997
Max: 1.000000


In [21]:
# ============================================================
# CELL 21 — MODEL SANITY CHECK
# ============================================================

set_seed(
    RANDOM_STATE
)


model = SiameseNetwork(
    embedding_dim=EMBEDDING_DIM
).to(
    DEVICE
)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print("=" * 70)
print("MODEL")
print("=" * 70)

print(model)

print()

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# REAL BATCH
# ============================================================

pair_batch, target_batch = next(
    iter(train_loader)
)


pair_batch = pair_batch.to(
    DEVICE,
    non_blocking=True,
)


target_batch = target_batch.to(
    DEVICE,
    non_blocking=True,
)


amp_enabled = (
    USE_AMP
    and
    DEVICE.type == "cuda"
)


with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.float16,
    enabled=amp_enabled,
):

    z1, z2 = model(
        pair_batch
    )


# Similarity in FP32

similarity = F.cosine_similarity(
    z1.float(),
    z2.float(),
    dim=1,
    eps=1e-8,
)


# ============================================================
# CHECK EMBEDDING NORMS
# ============================================================

z1_norm = (
    z1.float()
    .norm(
        p=2,
        dim=1,
    )
)


z2_norm = (
    z2.float()
    .norm(
        p=2,
        dim=1,
    )
)


print()
print("=" * 70)
print("FORWARD CHECK")
print("=" * 70)

print(
    "z1 shape:",
    z1.shape
)

print(
    "z2 shape:",
    z2.shape
)

print()

print(
    "z1 norm mean:",
    z1_norm.mean().item()
)

print(
    "z2 norm mean:",
    z2_norm.mean().item()
)

print()

print(
    "Similarity min:",
    similarity.min().item()
)

print(
    "Similarity max:",
    similarity.max().item()
)

print(
    "Similarity mean:",
    similarity.mean().item()
)


assert z1.shape == (
    BATCH_SIZE,
    EMBEDDING_DIM,
)

assert z2.shape == (
    BATCH_SIZE,
    EMBEDDING_DIM,
)


assert torch.allclose(
    z1_norm,
    torch.ones_like(z1_norm),
    atol=2e-3,
)


assert torch.allclose(
    z2_norm,
    torch.ones_like(z2_norm),
    atol=2e-3,
)


assert (
    similarity.min()
    >= -1.0001
)

assert (
    similarity.max()
    <= 1.0001
)


print()
print(
    "Model + embeddings + cosine: OK"
)

MODEL
SiameseNetwork(
  (encoder): FCGRCNNEncoder(
    (features): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU(inplace=True)
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): GroupNorm(8, 64, eps=1e-05, affine=True, bias=True)
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): GroupNorm(8, 128, eps=1e-05, affine=True, bias=True)
      (12): ReLU(inplace=True)
      (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (14): Conv2d(128, 128, kernel_size=

In [22]:
# ============================================================
# CELL 22 — EXPERIMENT DATALOADER FACTORY
# ============================================================

def build_experiment_loaders():

    # IMPORTANT:
    # recreate datasets from scratch so that every experiment
    # starts from the same RNG state and therefore sees the
    # same dynamic training-pair sequence.

    train_ds = SiamesePairDataset(

        metadata=train_metadata,

        fcgr_memmap=fcgr_memmap,

        id_to_fcgr_row=id_to_fcgr_row,

        pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH,

        positive_probability=POSITIVE_PAIR_PROBABILITY,

        seed=RANDOM_STATE,

        deterministic=False,
    )


    val_ds = SiamesePairDataset(

        metadata=val_pair_pool,

        fcgr_memmap=fcgr_memmap,

        id_to_fcgr_row=id_to_fcgr_row,

        pairs_per_epoch=VAL_PAIRS,

        positive_probability=POSITIVE_PAIR_PROBABILITY,

        seed=RANDOM_STATE + 10_000,

        deterministic=True,
    )


    test_ds = SiamesePairDataset(

        metadata=test_pair_pool,

        fcgr_memmap=fcgr_memmap,

        id_to_fcgr_row=id_to_fcgr_row,

        pairs_per_epoch=TEST_PAIRS,

        positive_probability=POSITIVE_PAIR_PROBABILITY,

        seed=RANDOM_STATE + 20_000,

        deterministic=True,
    )


    train_dl = DataLoader(

        train_ds,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        drop_last=False,
    )


    val_dl = DataLoader(

        val_ds,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        drop_last=False,
    )


    test_dl = DataLoader(

        test_ds,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        drop_last=False,
    )


    return (
        train_dl,
        val_dl,
        test_dl,
    )


print(
    "Experiment DataLoader factory: OK"
)

Experiment DataLoader factory: OK


In [23]:
# ============================================================
# CELL 23 — TRAINING COMPONENT FACTORY
# ============================================================

AMP_ENABLED = (
    USE_AMP
    and
    DEVICE.type == "cuda"
)


def build_training_components(
    margin,
):

    # --------------------------------------------------------
    # Same initialization for every experiment
    # --------------------------------------------------------

    set_seed(
        RANDOM_STATE
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = SiameseNetwork(
        embedding_dim=EMBEDDING_DIM
    ).to(
        DEVICE
    )


    # --------------------------------------------------------
    # AdamW
    # --------------------------------------------------------

    fused_adamw = False


    try:

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,

            fused=(
                DEVICE.type == "cuda"
            ),
        )

        fused_adamw = (
            DEVICE.type == "cuda"
        )


    except (
        TypeError,
        RuntimeError,
    ):

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,
        )


    # --------------------------------------------------------
    # AMP scaler
    # --------------------------------------------------------

    if hasattr(
        torch,
        "amp"
    ):

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=AMP_ENABLED,
        )

    else:

        scaler = torch.cuda.amp.GradScaler(
            enabled=AMP_ENABLED
        )


    # --------------------------------------------------------
    # Official CosineEmbeddingLoss wrapper
    # --------------------------------------------------------

    criterion = CosineMetricLoss(
        margin=margin
    )


    return (
        model,
        optimizer,
        scaler,
        criterion,
        fused_adamw,
    )


print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "LR:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Loss:",
    "CosineMetricLoss / nn.CosineEmbeddingLoss"
)

TRAINING CONFIGURATION
AMP: True
LR: 0.0005
Weight decay: 0.0001
Loss: CosineMetricLoss / nn.CosineEmbeddingLoss


In [24]:
# ============================================================
# CELL 24 — METRICS
# ============================================================

def safe_dprime(
    mean_a,
    std_a,
    mean_b,
    std_b,
):

    pooled_std = np.sqrt(
        (
            std_a ** 2
            +
            std_b ** 2
        )
        / 2.0
    )


    if pooled_std < 1e-12:
        return 0.0


    return (
        mean_a
        -
        mean_b
    ) / pooled_std


def compute_distribution_metrics(
    targets,
    similarities,
):

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float64,
    )


    positive_mask = (
        targets == 1
    )

    negative_mask = (
        targets == 0
    )


    cos_pos = similarities[
        positive_mask
    ]

    cos_neg = similarities[
        negative_mask
    ]


    # --------------------------------------------------------
    # Cosine statistics
    # --------------------------------------------------------

    cos_pos_mean = float(
        cos_pos.mean()
    )

    cos_neg_mean = float(
        cos_neg.mean()
    )


    cos_pos_std = float(
        cos_pos.std()
    )

    cos_neg_std = float(
        cos_neg.std()
    )


    cosine_gap = (
        cos_pos_mean
        -
        cos_neg_mean
    )


    cosine_dprime = safe_dprime(

        cos_pos_mean,
        cos_pos_std,

        cos_neg_mean,
        cos_neg_std,
    )


    # --------------------------------------------------------
    # Convert cosine -> Euclidean distance
    #
    # for unit embeddings:
    #
    # d² = 2 - 2 cos
    # --------------------------------------------------------

    distances = np.sqrt(
        np.clip(
            2.0
            -
            2.0 * similarities,
            a_min=0.0,
            a_max=None,
        )
    )


    dist_pos = distances[
        positive_mask
    ]

    dist_neg = distances[
        negative_mask
    ]


    d_pos = float(
        dist_pos.mean()
    )

    d_neg = float(
        dist_neg.mean()
    )


    d_pos_std = float(
        dist_pos.std()
    )

    d_neg_std = float(
        dist_neg.std()
    )


    distance_gap = (
        d_neg
        -
        d_pos
    )


    distance_dprime = safe_dprime(

        d_neg,
        d_neg_std,

        d_pos,
        d_pos_std,
    )


    return {

        "cos_pos":
            cos_pos_mean,

        "cos_neg":
            cos_neg_mean,

        "cos_gap":
            cosine_gap,

        "cos_dprime":
            cosine_dprime,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "distance_gap":
            distance_gap,

        "distance_dprime":
            distance_dprime,
    }

In [25]:
# ============================================================
# CELL 25 — VALIDATION THRESHOLD SEARCH
# ============================================================

def find_best_threshold(
    targets,
    similarities,
):

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float64,
    )


    # Higher cosine => positive pair.

    order = np.argsort(
        -similarities
    )


    y = targets[
        order
    ]

    scores = similarities[
        order
    ]


    positive = (
        y == 1
    ).astype(np.int64)

    negative = (
        y == 0
    ).astype(np.int64)


    tp = np.cumsum(
        positive
    )

    fp = np.cumsum(
        negative
    )


    total_positive = int(
        positive.sum()
    )

    total_negative = int(
        negative.sum()
    )


    fn = (
        total_positive
        -
        tp
    )

    tn = (
        total_negative
        -
        fp
    )


    # F1 for class 1

    f1_positive = (
        2.0 * tp
        /
        np.maximum(
            2.0 * tp
            +
            fp
            +
            fn,
            1,
        )
    )


    # F1 for class 0

    f1_negative = (
        2.0 * tn
        /
        np.maximum(
            2.0 * tn
            +
            fp
            +
            fn,
            1,
        )
    )


    macro_f1 = (
        f1_positive
        +
        f1_negative
    ) / 2.0


    # Only evaluate valid score boundaries.

    boundary_mask = np.r_[

        scores[:-1]
        !=
        scores[1:],

        True,
    ]


    valid_indices = np.where(
        boundary_mask
    )[0]


    best_relative_index = np.argmax(
        macro_f1[
            valid_indices
        ]
    )


    best_index = int(
        valid_indices[
            best_relative_index
        ]
    )


    best_threshold = float(
        scores[
            best_index
        ]
    )


    return best_threshold

In [26]:
# ============================================================
# CELL 26 — CLASSIFICATION METRICS
# ============================================================

def compute_classification_metrics(
    targets,
    similarities,
    threshold,
):

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float64,
    )


    predictions = (
        similarities
        >=
        threshold
    ).astype(
        np.int64
    )


    return {

        "roc_auc":
            float(
                roc_auc_score(
                    targets,
                    similarities,
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predictions,
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    targets,
                    predictions,
                    average="macro",
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predictions,
                )
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predictions,
                    zero_division=0,
                )
            ),

        "recall":
            float(
                recall_score(
                    targets,
                    predictions,
                    zero_division=0,
                )
            ),

        "confusion_matrix":
            confusion_matrix(
                targets,
                predictions,
            ),
    }

In [27]:
# ============================================================
# CELL 27 — OPTIMIZED EPOCH LOOP
# ============================================================

def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    collect_outputs=False,
):

    training = (
        optimizer is not None
    )


    if training:
        model.train()
    else:
        model.eval()


    # --------------------------------------------------------
    # GPU-side accumulators
    # --------------------------------------------------------

    loss_sum = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float32,
    )


    positive_similarity_sum = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float32,
    )


    negative_similarity_sum = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float32,
    )


    positive_count = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float32,
    )


    negative_count = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float32,
    )


    n_samples = 0


    all_targets = []

    all_similarities = []


    # Accurate epoch timing.

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


    start_time = time.perf_counter()


    context = (
        torch.enable_grad()
        if training
        else torch.inference_mode()
    )


    with context:

        for (
            pair_batch,
            target_batch,
        ) in loader:


            # ------------------------------------------------
            # CPU -> GPU
            # ------------------------------------------------

            pair_batch = pair_batch.to(

                DEVICE,

                non_blocking=True,
            )


            target_batch = target_batch.to(

                DEVICE,

                non_blocking=True,
            )


            batch_size = (
                target_batch.shape[0]
            )


            # ------------------------------------------------
            # TRAIN
            # ------------------------------------------------

            if training:

                optimizer.zero_grad(
                    set_to_none=True
                )


            # ------------------------------------------------
            # FORWARD
            # ------------------------------------------------

            with torch.autocast(

                device_type="cuda",

                dtype=torch.float16,

                enabled=AMP_ENABLED,
            ):

                z1, z2 = model(
                    pair_batch
                )


                loss, similarity = criterion(

                    z1,
                    z2,
                    target_batch,
                )


            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            if training:

                scaler.scale(
                    loss
                ).backward()


                scaler.step(
                    optimizer
                )


                scaler.update()


            # ------------------------------------------------
            # GPU STATISTICS
            # ------------------------------------------------

            similarity_detached = (
                similarity.detach()
            )


            target_detached = (
                target_batch.detach()
            )


            positive_mask = (
                target_detached
                >
                0.5
            )


            negative_mask = (
                ~positive_mask
            )


            loss_sum += (
                loss.detach()
                *
                batch_size
            )


            positive_similarity_sum += (
                similarity_detached[
                    positive_mask
                ]
                .sum()
            )


            negative_similarity_sum += (
                similarity_detached[
                    negative_mask
                ]
                .sum()
            )


            positive_count += (
                positive_mask
                .sum()
            )


            negative_count += (
                negative_mask
                .sum()
            )


            n_samples += (
                batch_size
            )


            # ------------------------------------------------
            # Only validation/test need full outputs.
            # ------------------------------------------------

            if collect_outputs:

                all_targets.append(

                    target_detached
                    .cpu()
                )


                all_similarities.append(

                    similarity_detached
                    .cpu()
                )


    # --------------------------------------------------------
    # Timing
    # --------------------------------------------------------

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    # --------------------------------------------------------
    # Single GPU -> CPU synchronization
    # --------------------------------------------------------

    epoch_loss = float(
        (
            loss_sum
            /
            max(
                n_samples,
                1,
            )
        ).item()
    )


    mean_positive_similarity = float(

        (
            positive_similarity_sum
            /
            positive_count.clamp_min(
                1
            )
        ).item()
    )


    mean_negative_similarity = float(

        (
            negative_similarity_sum
            /
            negative_count.clamp_min(
                1
            )
        ).item()
    )


    stats = {

        "loss":
            epoch_loss,

        "cos_pos":
            mean_positive_similarity,

        "cos_neg":
            mean_negative_similarity,

        "cos_gap":
            (
                mean_positive_similarity
                -
                mean_negative_similarity
            ),

        "seconds":
            elapsed,

        "pairs_per_second":
            (
                n_samples
                /
                max(
                    elapsed,
                    1e-9,
                )
            ),
    }


    # --------------------------------------------------------
    # Validation / test arrays
    # --------------------------------------------------------

    if collect_outputs:

        targets_np = (
            torch.cat(
                all_targets
            )
            .numpy()
        )


        similarities_np = (
            torch.cat(
                all_similarities
            )
            .numpy()
        )


        return (
            stats,
            targets_np,
            similarities_np,
        )


    return stats

In [28]:
# ============================================================
# CELL 28 — TRAIN ONE COSINE EXPERIMENT
# ============================================================

def train_cosine_experiment(
    margin,
    experiment_name,
    max_epochs=MAX_EPOCHS,
):

    print("=" * 80)
    print(
        f"EXPERIMENT: {experiment_name}"
    )
    print("=" * 80)

    print(
        "Cosine margin:",
        margin
    )

    print()


    # ========================================================
    # Clean previous GPU memory
    # ========================================================

    gc.collect()


    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()


    # ========================================================
    # Fresh DataLoaders
    # ========================================================

    (
        train_dl,
        val_dl,
        test_dl,
    ) = build_experiment_loaders()


    # ========================================================
    # Fresh model / optimizer
    # ========================================================

    (
        model,
        optimizer,
        scaler,
        criterion,
        fused_adamw,
    ) = build_training_components(
        margin
    )


    print(
        "Fused AdamW:",
        fused_adamw
    )

    print(
        "AMP:",
        AMP_ENABLED
    )

    print()


    # ========================================================
    # Experiment directory
    # ========================================================

    experiment_dir = (
        ARTIFACTS_DIR
        /
        experiment_name
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    checkpoint_path = (
        experiment_dir
        /
        "best_model.pt"
    )


    # ========================================================
    # Training state
    # ========================================================

    best_val_auc = -np.inf

    best_epoch = 0

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # Epoch loop
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        train_stats = run_epoch(

            model=model,

            loader=train_dl,

            criterion=criterion,

            optimizer=optimizer,

            scaler=scaler,

            collect_outputs=False,
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        (
            val_stats,
            val_targets,
            val_similarities,
        ) = run_epoch(

            model=model,

            loader=val_dl,

            criterion=criterion,

            optimizer=None,

            scaler=None,

            collect_outputs=True,
        )


        val_auc = float(

            roc_auc_score(

                val_targets,

                val_similarities,
            )
        )


        val_distribution = (
            compute_distribution_metrics(

                val_targets,

                val_similarities,
            )
        )


        # ----------------------------------------------------
        # Save history
        # ----------------------------------------------------

        epoch_result = {

            "epoch":
                epoch,

            "train_loss":
                train_stats["loss"],

            "val_loss":
                val_stats["loss"],

            "val_auc":
                val_auc,

            "train_cos_pos":
                train_stats["cos_pos"],

            "train_cos_neg":
                train_stats["cos_neg"],

            "train_cos_gap":
                train_stats["cos_gap"],

            "val_cos_pos":
                val_distribution["cos_pos"],

            "val_cos_neg":
                val_distribution["cos_neg"],

            "val_cos_gap":
                val_distribution["cos_gap"],

            "val_cos_dprime":
                val_distribution["cos_dprime"],

            "val_d_pos":
                val_distribution["d_pos"],

            "val_d_neg":
                val_distribution["d_neg"],

            "val_distance_gap":
                val_distribution[
                    "distance_gap"
                ],

            "val_distance_dprime":
                val_distribution[
                    "distance_dprime"
                ],

            "train_seconds":
                train_stats["seconds"],

            "val_seconds":
                val_stats["seconds"],

            "train_pairs_per_second":
                train_stats[
                    "pairs_per_second"
                ],
        }


        history.append(
            epoch_result
        )


        # ----------------------------------------------------
        # Checkpoint: validation ROC-AUC
        # ----------------------------------------------------

        improved = (

            val_auc
            >
            best_val_auc
            +
            MIN_DELTA
        )


        if improved:

            best_val_auc = (
                val_auc
            )

            best_epoch = (
                epoch
            )

            epochs_without_improvement = 0


            torch.save(

                {
                    "epoch":
                        epoch,

                    "margin":
                        margin,

                    "val_auc":
                        val_auc,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),
                },

                checkpoint_path,
            )


        else:

            epochs_without_improvement += 1


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        marker = (
            "  *BEST*"
            if improved
            else ""
        )


        print(

            f"Epoch {epoch:02d}/{max_epochs} | "

            f"train loss "
            f"{train_stats['loss']:.5f} | "

            f"val loss "
            f"{val_stats['loss']:.5f} | "

            f"val AUC "
            f"{val_auc:.5f} | "

            f"cos+ "
            f"{val_distribution['cos_pos']:.4f} | "

            f"cos- "
            f"{val_distribution['cos_neg']:.4f} | "

            f"gap "
            f"{val_distribution['cos_gap']:.4f} | "

            f"d' "
            f"{val_distribution['cos_dprime']:.3f} | "

            f"{train_stats['seconds']:.1f}s"

            f"{marker}"
        )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (
            epochs_without_improvement
            >=
            EARLY_STOPPING_PATIENCE
        ):

            print()
            print(
                "Early stopping."
            )

            break


    # ========================================================
    # Save history
    # ========================================================

    history_df = pd.DataFrame(
        history
    )


    history_df.to_csv(

        experiment_dir
        /
        "history.tsv",

        sep="\t",

        index=False,
    )


    print()
    print("=" * 80)

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best Val ROC-AUC:",
        f"{best_val_auc:.5f}"
    )

    print(
        "Checkpoint:",
        checkpoint_path
    )

    print("=" * 80)


    return {

        "model":
            model,

        "criterion":
            criterion,

        "train_loader":
            train_dl,

        "val_loader":
            val_dl,

        "test_loader":
            test_dl,

        "history":
            history_df,

        "checkpoint_path":
            checkpoint_path,

        "best_epoch":
            best_epoch,

        "best_val_auc":
            best_val_auc,

        "experiment_dir":
            experiment_dir,
    }

In [29]:
'''# ============================================================
# CELL 29 — REFERENCE COSINE RUN
# ============================================================

REFERENCE_EXPERIMENT_NAME = (
    "cosine_margin_0p21875"
)


reference_run = train_cosine_experiment(

    margin=REFERENCE_COSINE_MARGIN,

    experiment_name=(
        REFERENCE_EXPERIMENT_NAME
    ),

    max_epochs=MAX_EPOCHS,
)'''

'# ============================================================\n# CELL 29 — REFERENCE COSINE RUN\n# ============================================================\n\nREFERENCE_EXPERIMENT_NAME = (\n    "cosine_margin_0p21875"\n)\n\n\nreference_run = train_cosine_experiment(\n\n    margin=REFERENCE_COSINE_MARGIN,\n\n    experiment_name=(\n        REFERENCE_EXPERIMENT_NAME\n    ),\n\n    max_epochs=MAX_EPOCHS,\n)'

In [30]:
# ============================================================
# CELL 30 — COSINE ENCODER V1
#
# Minimal anti-collapse modification:
# final projection has NO bias.
#
# Everything else remains identical to Baseline C.
# ============================================================

class FCGRCNNEncoderCosineV1(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                32,
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                64,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            ),
        )


        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256,
            ),

            nn.ReLU(
                inplace=True
            ),

            # -----------------------------------------------
            # ONLY architectural change
            # -----------------------------------------------

            nn.Linear(
                256,
                embedding_dim,
                bias=False,
            ),
        )


    def forward(
        self,
        x,
    ):

        x = self.features(
            x
        )

        z = self.projection(
            x
        )


        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8,
        )


        return z

In [31]:
# ============================================================
# CELL 31 — SIAMESE COSINE V1
# ============================================================

class SiameseNetworkCosineV1(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        self.encoder = FCGRCNNEncoderCosineV1(
            embedding_dim=embedding_dim
        )


    def forward(
        self,
        pair_batch,
    ):

        batch_size = (
            pair_batch.shape[0]
        )


        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        z = self.encoder(
            x
        )


        z = z.reshape(
            batch_size,
            2,
            -1,
        )


        return (
            z[:, 0, :],
            z[:, 1, :],
        )

In [32]:
# ============================================================
# CELL 32 — INITIAL GEOMETRY COMPARISON
# ============================================================

def evaluate_initial_geometry(
    model_class,
    name,
    max_batches=30,
):

    set_seed(
        RANDOM_STATE
    )


    model = model_class(
        embedding_dim=EMBEDDING_DIM
    ).to(
        DEVICE
    )


    model.eval()


    similarities_list = []
    targets_list = []


    with torch.inference_mode():

        for batch_index, (
            pair_batch,
            target_batch,
        ) in enumerate(
            val_loader
        ):

            if batch_index >= max_batches:
                break


            pair_batch = pair_batch.to(
                DEVICE,
                non_blocking=True,
            )


            with torch.autocast(

                device_type=DEVICE.type,

                dtype=torch.float16,

                enabled=(
                    USE_AMP
                    and
                    DEVICE.type == "cuda"
                ),
            ):

                z1, z2 = model(
                    pair_batch
                )


            similarities = (
                F.cosine_similarity(

                    z1.float(),

                    z2.float(),

                    dim=1,

                    eps=1e-8,
                )
            )


            similarities_list.append(
                similarities.cpu()
            )


            targets_list.append(
                target_batch.cpu()
            )


    similarities = (
        torch.cat(
            similarities_list
        )
        .numpy()
    )


    targets = (
        torch.cat(
            targets_list
        )
        .numpy()
        .astype(int)
    )


    positive = (
        similarities[
            targets == 1
        ]
    )


    negative = (
        similarities[
            targets == 0
        ]
    )


    result = {

        "model":
            name,

        "mean":
            float(
                similarities.mean()
            ),

        "std":
            float(
                similarities.std()
            ),

        "min":
            float(
                similarities.min()
            ),

        "median":
            float(
                np.median(
                    similarities
                )
            ),

        "max":
            float(
                similarities.max()
            ),

        "cos_pos":
            float(
                positive.mean()
            ),

        "cos_neg":
            float(
                negative.mean()
            ),

        "gap":
            float(
                positive.mean()
                -
                negative.mean()
            ),

        "auc":
            float(
                roc_auc_score(
                    targets,
                    similarities,
                )
            ),
    }


    del model

    gc.collect()


    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


    return result


baseline_geometry = (
    evaluate_initial_geometry(

        SiameseNetwork,

        "Baseline head",
    )
)


v1_geometry = (
    evaluate_initial_geometry(

        SiameseNetworkCosineV1,

        "Cosine V1 bias=False",
    )
)


initial_geometry_df = pd.DataFrame(
    [
        baseline_geometry,
        v1_geometry,
    ]
)


display(
    initial_geometry_df
)

,model,mean,std,min,median,max,cos_pos,cos_neg,gap,auc
0,Baseline head,0.999979,0.000077,0.998193,0.999993,1.0,0.999979,0.999979,-5.960464e-08,0.54481
1,Cosine V1 bias=False,0.999978,0.000080,0.998110,0.999993,1.0,0.999978,0.999978,-1.192093e-07,0.54445


In [33]:
# ============================================================
# CELL 33 — COSINE ENCODER V2
#
# Anti-collapse projection:
#
# Linear -> ReLU -> Linear(bias=False)
#        -> LayerNorm(affine=False)
#        -> L2 normalize
#
# CNN backbone remains identical to Baseline C.
# ============================================================

class FCGRCNNEncoderCosineV2(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        # ====================================================
        # SAME CNN AS BASELINE C
        # ====================================================

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                32,
            ),

            nn.ReLU(
                inplace=True,
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                64,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True,
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            ),
        )


        # ====================================================
        # COSINE PROJECTION HEAD V2
        # ====================================================

        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.Linear(
                256,
                embedding_dim,
                bias=False,
            ),

            # ------------------------------------------------
            # Key V2 modification.
            #
            # Per-sample centering / scaling.
            # No learnable affine bias.
            # ------------------------------------------------

            nn.LayerNorm(
                embedding_dim,
                elementwise_affine=False,
            ),
        )


    def forward(
        self,
        x,
    ):

        x = self.features(
            x
        )


        z = self.projection(
            x
        )


        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8,
        )


        return z

In [34]:
# ============================================================
# CELL 34 — SIAMESE COSINE V2
# ============================================================

class SiameseNetworkCosineV2(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        self.encoder = (
            FCGRCNNEncoderCosineV2(
                embedding_dim=embedding_dim,
            )
        )


    def forward(
        self,
        pair_batch,
    ):

        batch_size = (
            pair_batch.shape[0]
        )


        # ----------------------------------------------------
        # [B, 2, 1, 64, 64]
        # ->
        # [2B, 1, 64, 64]
        # ----------------------------------------------------

        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        z = self.encoder(
            x
        )


        # ----------------------------------------------------
        # [2B, D]
        # ->
        # [B, 2, D]
        # ----------------------------------------------------

        z = z.reshape(
            batch_size,
            2,
            -1,
        )


        z1 = z[:, 0, :]

        z2 = z[:, 1, :]


        return z1, z2

In [35]:
# ============================================================
# CELL 35 — INITIAL GEOMETRY:
# BASELINE vs V1 vs V2
# ============================================================

baseline_geometry = (
    evaluate_initial_geometry(
        SiameseNetwork,
        "Baseline head",
    )
)


v1_geometry = (
    evaluate_initial_geometry(
        SiameseNetworkCosineV1,
        "V1 bias=False",
    )
)


v2_geometry = (
    evaluate_initial_geometry(
        SiameseNetworkCosineV2,
        "V2 LayerNorm",
    )
)


geometry_comparison_df = pd.DataFrame(
    [
        baseline_geometry,
        v1_geometry,
        v2_geometry,
    ]
)


display(
    geometry_comparison_df
)

,model,mean,std,min,median,max,cos_pos,cos_neg,gap,auc
0,Baseline head,0.999979,0.000077,0.998193,0.999993,1.0,0.999979,0.999979,-5.960464e-08,0.544810
1,V1 bias=False,0.999978,0.000080,0.998110,0.999993,1.0,0.999978,0.999978,-1.192093e-07,0.544450
2,V2 LayerNorm,0.999978,0.000082,0.998096,0.999993,1.0,0.999978,0.999978,-5.960464e-08,0.544683


In [36]:
# ============================================================
# CELL 36 — INITIAL GEOMETRY: FP32 vs AMP
# ============================================================

def evaluate_precision_geometry(
    use_amp,
    max_batches=30,
):

    set_seed(RANDOM_STATE)

    model = SiameseNetwork(
        embedding_dim=EMBEDDING_DIM
    ).to(DEVICE)

    model.eval()

    similarities_list = []
    targets_list = []

    with torch.inference_mode():

        for batch_index, (
            pair_batch,
            target_batch,
        ) in enumerate(val_loader):

            if batch_index >= max_batches:
                break

            pair_batch = pair_batch.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(
                    use_amp
                    and
                    DEVICE.type == "cuda"
                ),
            ):

                z1, z2 = model(
                    pair_batch
                )

            similarity = F.cosine_similarity(
                z1.float(),
                z2.float(),
                dim=1,
                eps=1e-8,
            )

            similarities_list.append(
                similarity.cpu()
            )

            targets_list.append(
                target_batch.cpu()
            )

    similarities = (
        torch.cat(similarities_list)
        .numpy()
    )

    targets = (
        torch.cat(targets_list)
        .numpy()
        .astype(int)
    )

    positive = similarities[
        targets == 1
    ]

    negative = similarities[
        targets == 0
    ]

    result = {
        "precision":
            "AMP FP16" if use_amp else "FP32",

        "mean":
            float(similarities.mean()),

        "std":
            float(similarities.std()),

        "min":
            float(similarities.min()),

        "median":
            float(np.median(similarities)),

        "max":
            float(similarities.max()),

        "cos_pos":
            float(positive.mean()),

        "cos_neg":
            float(negative.mean()),

        "gap":
            float(
                positive.mean()
                -
                negative.mean()
            ),

        "auc":
            float(
                roc_auc_score(
                    targets,
                    similarities,
                )
            ),
    }

    del model
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


precision_comparison_df = pd.DataFrame(
    [
        evaluate_precision_geometry(
            use_amp=False
        ),

        evaluate_precision_geometry(
            use_amp=True
        ),
    ]
)


display(
    precision_comparison_df
)

,precision,mean,std,min,median,max,cos_pos,cos_neg,gap,auc
0,FP32,0.999979,0.000078,0.998168,0.999993,1.0,0.999979,0.999979,-1.192093e-07,0.544611
1,AMP FP16,0.999979,0.000077,0.998193,0.999993,1.0,0.999979,0.999979,-5.960464e-08,0.544810


In [37]:
# ============================================================
# CELL 37 — STAGE-WISE GEOMETRY DIAGNOSTIC
# ============================================================

set_seed(RANDOM_STATE)

stage_model = FCGRCNNEncoder(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)

stage_model.eval()


stage_values = {
    "cnn_flat": [],
    "hidden_linear": [],
    "hidden_relu": [],
    "raw_embedding": [],
    "normalized_embedding": [],
}


MAX_STAGE_BATCHES = 20


def paired_cosine(
    x,
    batch_size,
):

    x = x.reshape(
        batch_size,
        2,
        -1,
    )

    x1 = x[:, 0, :]
    x2 = x[:, 1, :]

    return F.cosine_similarity(
        x1.float(),
        x2.float(),
        dim=1,
        eps=1e-8,
    )


with torch.inference_mode():

    for batch_index, (
        pair_batch,
        _,
    ) in enumerate(val_loader):

        if batch_index >= MAX_STAGE_BATCHES:
            break

        batch_size = (
            pair_batch.shape[0]
        )

        pair_batch = pair_batch.to(
            DEVICE,
            non_blocking=True,
        )

        # Deliberately FP32 for diagnosis.
        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        # ----------------------------------------------------
        # CNN
        # ----------------------------------------------------

        features = stage_model.features(
            x
        )

        flat = torch.flatten(
            features,
            start_dim=1,
        )


        # ----------------------------------------------------
        # Projection head, explicitly stage by stage
        # ----------------------------------------------------

        hidden_linear = (
            stage_model.projection[1](
                flat
            )
        )

        hidden_relu = (
            stage_model.projection[2](
                hidden_linear
            )
        )

        raw_embedding = (
            stage_model.projection[3](
                hidden_relu
            )
        )

        normalized_embedding = F.normalize(
            raw_embedding,
            p=2,
            dim=1,
            eps=1e-8,
        )


        tensors = {
            "cnn_flat":
                flat,

            "hidden_linear":
                hidden_linear,

            "hidden_relu":
                hidden_relu,

            "raw_embedding":
                raw_embedding,

            "normalized_embedding":
                normalized_embedding,
        }


        for name, tensor in tensors.items():

            similarities = paired_cosine(
                tensor,
                batch_size,
            )

            stage_values[name].append(
                similarities.cpu()
            )


# ============================================================
# SUMMARY
# ============================================================

stage_results = []


for name, chunks in stage_values.items():

    similarities = (
        torch.cat(chunks)
        .numpy()
    )

    stage_results.append(
        {
            "stage":
                name,

            "mean_cosine":
                float(
                    similarities.mean()
                ),

            "std_cosine":
                float(
                    similarities.std()
                ),

            "min_cosine":
                float(
                    similarities.min()
                ),

            "median_cosine":
                float(
                    np.median(
                        similarities
                    )
                ),

            "max_cosine":
                float(
                    similarities.max()
                ),
        }
    )


stage_geometry_df = pd.DataFrame(
    stage_results
)


display(
    stage_geometry_df
)


del stage_model

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

,stage,mean_cosine,std_cosine,min_cosine,median_cosine,max_cosine
0,cnn_flat,0.999978,0.000087,0.998215,0.999993,1.0
1,hidden_linear,0.999975,0.000096,0.997966,0.999993,1.0
2,hidden_relu,0.999975,0.000096,0.997966,0.999993,1.0
3,raw_embedding,0.999978,0.000084,0.998202,0.999993,1.0
4,normalized_embedding,0.999978,0.000084,0.998202,0.999993,1.0


In [38]:
# ============================================================
# CELL 38 — RAW FCGR GEOMETRY DIAGNOSTIC
#
# Goal:
# determine whether the strong common direction already exists
# in the k=6 FCGR inputs, before the CNN.
# ============================================================

raw_cosine_chunks = []
centered_cosine_chunks = []
target_chunks = []

MAX_RAW_BATCHES = 30


with torch.inference_mode():

    for batch_index, (
        pair_batch,
        target_batch,
    ) in enumerate(val_loader):

        if batch_index >= MAX_RAW_BATCHES:
            break


        # pair_batch:
        # [B, 2, 1, 64, 64]

        pair_batch = pair_batch.float()


        batch_size = pair_batch.shape[0]


        # ====================================================
        # RAW FCGR
        # ====================================================

        x1 = pair_batch[:, 0].flatten(
            start_dim=1
        )

        x2 = pair_batch[:, 1].flatten(
            start_dim=1
        )


        raw_cosine = F.cosine_similarity(
            x1,
            x2,
            dim=1,
            eps=1e-8,
        )


        # ====================================================
        # DIAGNOSTIC COMMON-COMPONENT REMOVAL
        #
        # Compute mean FCGR across all 2B samples in the batch.
        #
        # IMPORTANT:
        # this is only a diagnostic, NOT yet preprocessing.
        # ====================================================

        all_fcgr = pair_batch.reshape(
            batch_size * 2,
            -1,
        )


        common_component = all_fcgr.mean(
            dim=0,
            keepdim=True,
        )


        x1_centered = (
            x1
            -
            common_component
        )


        x2_centered = (
            x2
            -
            common_component
        )


        centered_cosine = F.cosine_similarity(
            x1_centered,
            x2_centered,
            dim=1,
            eps=1e-8,
        )


        raw_cosine_chunks.append(
            raw_cosine
        )

        centered_cosine_chunks.append(
            centered_cosine
        )

        target_chunks.append(
            target_batch
        )


# ============================================================
# CONCATENATE
# ============================================================

raw_cosine = torch.cat(
    raw_cosine_chunks
).numpy()


centered_cosine = torch.cat(
    centered_cosine_chunks
).numpy()


targets = (
    torch.cat(
        target_chunks
    )
    .numpy()
    .astype(int)
)


# ============================================================
# SUMMARY FUNCTION
# ============================================================

def summarize_input_geometry(
    name,
    similarities,
    targets,
):

    positive = similarities[
        targets == 1
    ]

    negative = similarities[
        targets == 0
    ]


    return {

        "representation":
            name,

        "mean":
            float(
                similarities.mean()
            ),

        "std":
            float(
                similarities.std()
            ),

        "min":
            float(
                similarities.min()
            ),

        "median":
            float(
                np.median(
                    similarities
                )
            ),

        "max":
            float(
                similarities.max()
            ),

        "cos_pos":
            float(
                positive.mean()
            ),

        "cos_neg":
            float(
                negative.mean()
            ),

        "gap":
            float(
                positive.mean()
                -
                negative.mean()
            ),

        "auc":
            float(
                roc_auc_score(
                    targets,
                    similarities,
                )
            ),
    }


raw_geometry = summarize_input_geometry(
    "Raw FCGR",
    raw_cosine,
    targets,
)


centered_geometry = summarize_input_geometry(
    "FCGR - common component",
    centered_cosine,
    targets,
)


input_geometry_df = pd.DataFrame(
    [
        raw_geometry,
        centered_geometry,
    ]
)


display(
    input_geometry_df
)

,representation,mean,std,min,median,max,cos_pos,cos_neg,gap,auc
0,Raw FCGR,0.146689,0.125572,0.000000,0.114744,0.950246,0.153469,0.139433,0.014035,0.516279
1,FCGR - common component,0.008114,0.082143,-0.431066,-0.004093,0.782943,0.014796,0.000963,0.013833,0.549318


In [39]:
# ============================================================
# CELL 39 — FIRST CNN BLOCK GEOMETRY
# ============================================================

set_seed(RANDOM_STATE)

diagnostic_encoder = FCGRCNNEncoder(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)

diagnostic_encoder.eval()


stage_chunks = {
    "raw_fcgr": [],
    "conv1": [],
    "groupnorm1": [],
    "relu1": [],
    "conv2": [],
    "relu2": [],
    "pool1": [],
}


MAX_BATCHES = 20


def pairwise_stage_cosine(
    tensor,
    batch_size,
):

    tensor = tensor.reshape(
        batch_size,
        2,
        -1,
    )

    return F.cosine_similarity(
        tensor[:, 0].float(),
        tensor[:, 1].float(),
        dim=1,
        eps=1e-8,
    )


with torch.inference_mode():

    for batch_index, (
        pair_batch,
        _,
    ) in enumerate(val_loader):

        if batch_index >= MAX_BATCHES:
            break


        batch_size = pair_batch.shape[0]


        pair_batch = pair_batch.to(
            DEVICE,
            non_blocking=True,
        )


        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        # Raw input

        stage_chunks[
            "raw_fcgr"
        ].append(
            pairwise_stage_cosine(
                x,
                batch_size,
            ).cpu()
        )


        # ----------------------------------------------------
        # First CNN block
        # ----------------------------------------------------

        conv1 = (
            diagnostic_encoder
            .features[0](x)
        )


        gn1 = (
            diagnostic_encoder
            .features[1](conv1)
        )


        relu1 = (
            diagnostic_encoder
            .features[2](gn1)
        )


        conv2 = (
            diagnostic_encoder
            .features[3](relu1)
        )


        relu2 = (
            diagnostic_encoder
            .features[4](conv2)
        )


        pool1 = (
            diagnostic_encoder
            .features[5](relu2)
        )


        tensors = {
            "conv1":
                conv1,

            "groupnorm1":
                gn1,

            "relu1":
                relu1,

            "conv2":
                conv2,

            "relu2":
                relu2,

            "pool1":
                pool1,
        }


        for name, tensor in tensors.items():

            stage_chunks[
                name
            ].append(

                pairwise_stage_cosine(
                    tensor,
                    batch_size,
                ).cpu()
            )


# ============================================================
# SUMMARY
# ============================================================

first_block_results = []


for name, chunks in stage_chunks.items():

    sims = (
        torch.cat(chunks)
        .numpy()
    )


    first_block_results.append(
        {
            "stage":
                name,

            "mean_cosine":
                float(sims.mean()),

            "std_cosine":
                float(sims.std()),

            "min_cosine":
                float(sims.min()),

            "median_cosine":
                float(np.median(sims)),

            "max_cosine":
                float(sims.max()),
        }
    )


first_block_df = pd.DataFrame(
    first_block_results
)


display(
    first_block_df
)


del diagnostic_encoder

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

,stage,mean_cosine,std_cosine,min_cosine,median_cosine,max_cosine
0,raw_fcgr,0.147286,0.125416,0.000000,0.115842,0.897374
1,conv1,0.999989,0.000019,0.999633,0.999993,1.000000
2,groupnorm1,0.999967,0.000040,0.999413,0.999977,1.000000
3,relu1,0.999967,0.000040,0.999413,0.999977,1.000000
4,conv2,0.999984,0.000019,0.999738,0.999988,1.000000
5,relu2,0.999984,0.000019,0.999738,0.999988,1.000000
6,pool1,0.999980,0.000032,0.999494,0.999989,1.000000


In [43]:
# ============================================================
# CELL 39b — CONV1 BIAS ABLATION
#
# Same Conv1 weights:
#   1) normal output with bias
#   2) output with bias manually removed
#
# This isolates the effect of Conv1 bias.
# ============================================================

set_seed(RANDOM_STATE)

diagnostic_encoder = FCGRCNNEncoder(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)

diagnostic_encoder.eval()


with_bias_chunks = []
without_bias_chunks = []
raw_chunks = []

MAX_BATCHES = 20


with torch.inference_mode():

    for batch_index, (
        pair_batch,
        _
    ) in enumerate(val_loader):

        if batch_index >= MAX_BATCHES:
            break


        batch_size = pair_batch.shape[0]

        pair_batch = pair_batch.to(
            DEVICE,
            non_blocking=True,
        )


        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        conv1_layer = (
            diagnostic_encoder.features[0]
        )


        # ====================================================
        # NORMAL CONV1
        # ====================================================

        conv_with_bias = conv1_layer(
            x
        )


        # ====================================================
        # SAME WEIGHTS — NO BIAS
        # ====================================================

        conv_without_bias = F.conv2d(

            x,

            weight=conv1_layer.weight,

            bias=None,

            stride=conv1_layer.stride,

            padding=conv1_layer.padding,

            dilation=conv1_layer.dilation,

            groups=conv1_layer.groups,
        )


        # ====================================================
        # PAIRWISE COSINE
        # ====================================================

        raw_chunks.append(

            pairwise_stage_cosine(
                x,
                batch_size,
            ).cpu()
        )


        with_bias_chunks.append(

            pairwise_stage_cosine(
                conv_with_bias,
                batch_size,
            ).cpu()
        )


        without_bias_chunks.append(

            pairwise_stage_cosine(
                conv_without_bias,
                batch_size,
            ).cpu()
        )


def summarize_cosine(
    name,
    chunks,
):

    values = (
        torch.cat(chunks)
        .numpy()
    )


    return {

        "representation":
            name,

        "mean_cosine":
            float(values.mean()),

        "std_cosine":
            float(values.std()),

        "min_cosine":
            float(values.min()),

        "median_cosine":
            float(np.median(values)),

        "max_cosine":
            float(values.max()),
    }


conv1_bias_test_df = pd.DataFrame(
    [
        summarize_cosine(
            "Raw FCGR",
            raw_chunks,
        ),

        summarize_cosine(
            "Conv1 with bias",
            with_bias_chunks,
        ),

        summarize_cosine(
            "Conv1 same weights, bias removed",
            without_bias_chunks,
        ),
    ]
)


display(
    conv1_bias_test_df
)


# ============================================================
# EXTRA: BIAS vs SIGNAL MAGNITUDE
# ============================================================

print()
print("=" * 70)
print("CONV1 MAGNITUDE CHECK")
print("=" * 70)

print(
    "Input abs mean:",
    float(x.abs().mean())
)

print(
    "Input abs max:",
    float(x.abs().max())
)

print()

print(
    "Conv signal abs mean (no bias):",
    float(
        conv_without_bias
        .abs()
        .mean()
    )
)

print(
    "Conv1 bias abs mean:",
    float(
        conv1_layer.bias
        .abs()
        .mean()
    )
)

print(
    "Conv1 bias abs max:",
    float(
        conv1_layer.bias
        .abs()
        .max()
    )
)


del diagnostic_encoder

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

,representation,mean_cosine,std_cosine,min_cosine,median_cosine,max_cosine
0,Raw FCGR,0.147286,0.125416,0.000000,0.115842,0.897374
1,Conv1 with bias,0.999989,0.000019,0.999633,0.999993,1.000000
2,"Conv1 same weights, bias removed",0.157613,0.126908,-0.003911,0.128089,0.907671



CONV1 MAGNITUDE CHECK
Input abs mean: 0.000244140625
Input abs max: 0.2222222238779068

Conv signal abs mean (no bias): 0.0002732232096605003
Conv1 bias abs mean: 0.16031858325004578
Conv1 bias abs max: 0.32862916588783264


C:\Users\simon\AppData\Local\Temp\ipykernel_17996\3957375863.py:211: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  float(


In [44]:
# ============================================================
# CELL 40 — COSINE V3
# BIAS-FREE CNN
# ============================================================

class FCGRCNNEncoderCosineV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()


        self.features = nn.Sequential(

            # ==================================================
            # 64 x 64
            # ==================================================

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                32,
            ),

            nn.ReLU(
                inplace=True,
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            # ==================================================
            # 32 x 32
            # ==================================================

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                64,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            # ==================================================
            # 16 x 16
            # ==================================================

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.MaxPool2d(2),


            # ==================================================
            # 8 x 8
            # ==================================================

            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                128,
            ),

            nn.ReLU(
                inplace=True,
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            ),
        )


        # ====================================================
        # SAME PROJECTION AS BASELINE C
        # ====================================================

        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256,
            ),

            nn.ReLU(
                inplace=True,
            ),

            nn.Linear(
                256,
                embedding_dim,
            ),
        )


    def forward(
        self,
        x,
    ):

        x = self.features(
            x
        )

        z = self.projection(
            x
        )

        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8,
        )

        return z

In [45]:
# ============================================================
# CELL 41 — SIAMESE COSINE V3
# ============================================================

class SiameseNetworkCosineV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderCosineV3(
                embedding_dim=embedding_dim,
            )
        )


    def forward(
        self,
        pair_batch,
    ):

        batch_size = (
            pair_batch.shape[0]
        )


        x = pair_batch.reshape(
            batch_size * 2,
            1,
            64,
            64,
        )


        z = self.encoder(
            x
        )


        z = z.reshape(
            batch_size,
            2,
            -1,
        )


        return (
            z[:, 0],
            z[:, 1],
        )

In [46]:
# ============================================================
# CELL 42 — INITIAL GEOMETRY:
# BASELINE vs BIAS-FREE CNN
# ============================================================

baseline_geometry = (
    evaluate_initial_geometry(
        SiameseNetwork,
        "Baseline CNN",
    )
)


v3_geometry = (
    evaluate_initial_geometry(
        SiameseNetworkCosineV3,
        "V3 conv bias=False",
    )
)


bias_geometry_df = pd.DataFrame(
    [
        baseline_geometry,
        v3_geometry,
    ]
)


display(
    bias_geometry_df
)

,model,mean,std,min,median,max,cos_pos,cos_neg,gap,auc
0,Baseline CNN,0.999979,0.000077,0.998193,0.999993,1.000000,0.999979,0.999979,-5.960464e-08,0.54481
1,V3 conv bias=False,0.944419,0.045570,0.462310,0.958633,0.994376,0.945797,0.942944,2.852559e-03,0.55091


In [47]:
# ============================================================
# CELL 43 — TRAINING COMPONENT FACTORY — COSINE V3
#
# V3:
# - convolution bias=False
# - GroupNorm
# - same projection head
# - official CosineEmbeddingLoss
# ============================================================

def build_training_components_v3(
    margin,
):

    # --------------------------------------------------------
    # Reproducible initialization
    # --------------------------------------------------------

    set_seed(
        RANDOM_STATE
    )


    # --------------------------------------------------------
    # COSINE V3 MODEL
    # --------------------------------------------------------

    model = SiameseNetworkCosineV3(
        embedding_dim=EMBEDDING_DIM
    ).to(
        DEVICE
    )


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    fused_adamw = False


    try:

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,

            fused=(
                DEVICE.type == "cuda"
            ),
        )


        fused_adamw = (
            DEVICE.type == "cuda"
        )


    except (
        TypeError,
        RuntimeError,
    ):

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,
        )


    # --------------------------------------------------------
    # AMP
    # --------------------------------------------------------

    amp_enabled = (
        USE_AMP
        and
        DEVICE.type == "cuda"
    )


    if hasattr(
        torch,
        "amp"
    ):

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=amp_enabled,
        )

    else:

        scaler = torch.cuda.amp.GradScaler(
            enabled=amp_enabled
        )


    # --------------------------------------------------------
    # OFFICIAL COSINE LOSS WRAPPER
    # --------------------------------------------------------

    criterion = CosineMetricLoss(
        margin=margin
    )


    return (
        model,
        optimizer,
        scaler,
        criterion,
        fused_adamw,
    )


print("=" * 70)
print("COSINE V3 TRAINING CONFIGURATION")
print("=" * 70)

print(
    "Model:",
    "SiameseNetworkCosineV3"
)

print(
    "Conv bias:",
    False
)

print(
    "LR:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Cosine margin:",
    REFERENCE_COSINE_MARGIN
)

COSINE V3 TRAINING CONFIGURATION
Model: SiameseNetworkCosineV3
Conv bias: False
LR: 0.0005
Weight decay: 0.0001
AMP: True
Cosine margin: 0.21875


In [49]:
import inspect

print(
    inspect.signature(
        run_epoch
    )
)

(model, loader, criterion, optimizer=None, scaler=None, collect_outputs=False)


In [51]:
# ============================================================
# DEBUG — CHECK run_epoch RETURN STRUCTURE
# ============================================================

debug_result = run_epoch(
    model=smoke_model,
    loader=smoke_val_loader,
    criterion=smoke_criterion,
    optimizer=None,
    scaler=None,
    collect_outputs=True,
)

print("Type:", type(debug_result))
print("Length:", len(debug_result))

for i, value in enumerate(debug_result):

    print()
    print(f"Element {i}")
    print("Type:", type(value))

    if hasattr(value, "shape"):
        print("Shape:", value.shape)

    elif isinstance(value, (list, tuple)):
        print("Length:", len(value))

    else:
        print("Value:", value)

Type: <class 'tuple'>
Length: 3

Element 0
Type: <class 'dict'>
Value: {'loss': 0.3545454740524292, 'cos_pos': 0.7070900797843933, 'cos_neg': 0.6321941018104553, 'cos_gap': 0.07489597797393799, 'seconds': 3.3548801000015374, 'pairs_per_second': 2980.732455981189}

Element 1
Type: <class 'numpy.ndarray'>
Shape: (10000,)

Element 2
Type: <class 'numpy.ndarray'>
Shape: (10000,)


In [52]:
# ============================================================
# CELL 44 — COSINE V3 SMOKE TEST
#
# 5 epochs from scratch.
#
# Goal:
# verify that the bias-free CNN avoids cosine collapse
# during optimization.
# ============================================================

SMOKE_EPOCHS = 5


# ============================================================
# FRESH LOADERS
# ============================================================

(
    smoke_train_loader,
    smoke_val_loader,
    _
) = build_experiment_loaders()


# ============================================================
# FRESH V3 MODEL
# ============================================================

(
    smoke_model,
    smoke_optimizer,
    smoke_scaler,
    smoke_criterion,
    smoke_fused_adamw,
) = build_training_components_v3(
    margin=REFERENCE_COSINE_MARGIN
)


print("=" * 70)
print("COSINE V3 — 5 EPOCH SMOKE TEST")
print("=" * 70)

print(
    "Margin:",
    REFERENCE_COSINE_MARGIN
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Fused AdamW:",
    smoke_fused_adamw
)

print(
    "AMP:",
    AMP_ENABLED
)

print()


# ============================================================
# STORAGE
# ============================================================

smoke_history = []


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(
    1,
    SMOKE_EPOCHS + 1,
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_output = run_epoch(

        model=smoke_model,

        loader=smoke_train_loader,

        criterion=smoke_criterion,

        optimizer=smoke_optimizer,

        scaler=smoke_scaler,

        collect_outputs=False,
    )


    # run_epoch may return either:
    # metrics_dict
    # OR
    # (metrics_dict, ...)
    if isinstance(
        train_output,
        tuple,
    ):

        train_metrics = (
            train_output[0]
        )

    else:

        train_metrics = (
            train_output
        )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_output = run_epoch(

        model=smoke_model,

        loader=smoke_val_loader,

        criterion=smoke_criterion,

        optimizer=None,

        scaler=None,

        collect_outputs=True,
    )


    # Exact structure verified experimentally:
    #
    # (
    #     metrics_dict,
    #     targets,
    #     similarities,
    # )

    (
        val_metrics,
        val_targets,
        val_scores,
    ) = val_output


    val_targets = np.asarray(
        val_targets
    ).astype(
        np.int64
    )


    val_scores = np.asarray(
        val_scores
    )


    # ========================================================
    # ROC-AUC
    # ========================================================

    val_auc = roc_auc_score(
        val_targets,
        val_scores,
    )


    # ========================================================
    # POSITIVE / NEGATIVE DISTRIBUTIONS
    # ========================================================

    positive_scores = (
        val_scores[
            val_targets == 1
        ]
    )


    negative_scores = (
        val_scores[
            val_targets == 0
        ]
    )


    cos_pos = float(
        positive_scores.mean()
    )


    cos_neg = float(
        negative_scores.mean()
    )


    gap = (
        cos_pos
        -
        cos_neg
    )


    # ========================================================
    # D-PRIME
    # ========================================================

    pooled_std = np.sqrt(

        0.5
        *
        (
            positive_scores.var()
            +
            negative_scores.var()
        )

        +

        1e-12
    )


    d_prime = float(
        gap
        /
        pooled_std
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    epoch_result = {

        "epoch":
            epoch,

        "train_loss":
            float(
                train_metrics["loss"]
            ),

        "val_loss":
            float(
                val_metrics["loss"]
            ),

        "val_auc":
            float(
                val_auc
            ),

        "cos_pos":
            cos_pos,

        "cos_neg":
            cos_neg,

        "gap":
            gap,

        "d_prime":
            d_prime,

        "train_seconds":
            float(
                train_metrics.get(
                    "seconds",
                    np.nan,
                )
            ),

        "val_seconds":
            float(
                val_metrics.get(
                    "seconds",
                    np.nan,
                )
            ),
    }


    smoke_history.append(
        epoch_result
    )


    # ========================================================
    # PRINT
    # ========================================================

    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{epoch_result['train_loss']:.5f}"

        f" | val loss "
        f"{epoch_result['val_loss']:.5f}"

        f" | val AUC "
        f"{epoch_result['val_auc']:.5f}"

        f" | cos+ "
        f"{epoch_result['cos_pos']:.4f}"

        f" | cos- "
        f"{epoch_result['cos_neg']:.4f}"

        f" | gap "
        f"{epoch_result['gap']:.4f}"

        f" | d' "
        f"{epoch_result['d_prime']:.3f}"

        f" | "
        f"{epoch_result['train_seconds']:.1f}s"
    )


# ============================================================
# FINAL TABLE
# ============================================================

smoke_history_df = pd.DataFrame(
    smoke_history
)


display(
    smoke_history_df
)

COSINE V3 — 5 EPOCH SMOKE TEST
Margin: 0.21875
Learning rate: 0.0005
Fused AdamW: True
AMP: True

Epoch 01/5 | train loss 0.37235 | val loss 0.35455 | val AUC 0.56529 | cos+ 0.7071 | cos- 0.6322 | gap 0.0749 | d' 0.213 | 32.0s
Epoch 02/5 | train loss 0.35482 | val loss 0.35181 | val AUC 0.56807 | cos+ 0.7088 | cos- 0.6336 | gap 0.0752 | d' 0.220 | 61.2s
Epoch 03/5 | train loss 0.33966 | val loss 0.34480 | val AUC 0.59479 | cos+ 0.7213 | cos- 0.6312 | gap 0.0902 | d' 0.287 | 163.3s
Epoch 04/5 | train loss 0.33771 | val loss 0.33705 | val AUC 0.60672 | cos+ 0.7211 | cos- 0.6095 | gap 0.1115 | d' 0.350 | 109.4s
Epoch 05/5 | train loss 0.33381 | val loss 0.33568 | val AUC 0.60843 | cos+ 0.6941 | cos- 0.5835 | gap 0.1106 | d' 0.352 | 43.8s


,epoch,train_loss,val_loss,val_auc,cos_pos,cos_neg,gap,d_prime,train_seconds,val_seconds
0,1,0.372350,0.354545,0.565292,0.707090,0.632194,0.074896,0.212694,31.987483,2.805566
1,2,0.354823,0.351815,0.568068,0.708776,0.633582,0.075195,0.219611,61.184255,13.099535
2,3,0.339662,0.344799,0.594787,0.721335,0.631156,0.090179,0.287174,163.301904,19.712824
3,4,0.337710,0.337051,0.606724,0.721056,0.609540,0.111515,0.350217,109.377744,4.850561
4,5,0.333808,0.335680,0.608434,0.694087,0.583474,0.110613,0.352438,43.759265,4.934352


In [53]:
# ============================================================
# CELL 45 — FULL COSINE V3 TRAINING
#
# Reference configuration:
# - bias-free CNN
# - GroupNorm
# - official CosineEmbeddingLoss
# - cosine margin = 0.21875
# - checkpoint on validation ROC-AUC
# - early stopping
# ============================================================


def train_cosine_v3_experiment(
    margin,
    experiment_name,
):

    # ========================================================
    # FRESH DATA LOADERS
    # ========================================================

    (
        train_loader_exp,
        val_loader_exp,
        test_loader_exp,
    ) = build_experiment_loaders()


    # ========================================================
    # FRESH MODEL / OPTIMIZER / LOSS
    # ========================================================

    (
        model,
        optimizer,
        scaler,
        criterion,
        fused_adamw,
    ) = build_training_components_v3(
        margin=margin
    )


    print("=" * 70)
    print(
        "EXPERIMENT:",
        experiment_name
    )
    print("=" * 70)

    print(
        "Cosine margin:",
        margin
    )

    print(
        "Learning rate:",
        LEARNING_RATE
    )

    print(
        "Fused AdamW:",
        fused_adamw
    )

    print(
        "AMP:",
        AMP_ENABLED
    )

    print(
        "Max epochs:",
        MAX_EPOCHS
    )

    print(
        "Early stopping patience:",
        EARLY_STOPPING_PATIENCE
    )

    print()


    # ========================================================
    # TRACKING
    # ========================================================

    history = []

    best_val_auc = -np.inf

    best_epoch = 0

    best_state = None

    epochs_without_improvement = 0


    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        train_output = run_epoch(

            model=model,

            loader=train_loader_exp,

            criterion=criterion,

            optimizer=optimizer,

            scaler=scaler,

            collect_outputs=False,
        )


        if isinstance(
            train_output,
            tuple,
        ):

            train_metrics = (
                train_output[0]
            )

        else:

            train_metrics = (
                train_output
            )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        (
            val_metrics,
            val_targets,
            val_scores,
        ) = run_epoch(

            model=model,

            loader=val_loader_exp,

            criterion=criterion,

            optimizer=None,

            scaler=None,

            collect_outputs=True,
        )


        val_targets = np.asarray(
            val_targets
        ).astype(
            np.int64
        )


        val_scores = np.asarray(
            val_scores
        )


        # ====================================================
        # VALIDATION METRICS
        # ====================================================

        val_auc = float(
            roc_auc_score(
                val_targets,
                val_scores,
            )
        )


        positive_scores = (
            val_scores[
                val_targets == 1
            ]
        )


        negative_scores = (
            val_scores[
                val_targets == 0
            ]
        )


        cos_pos = float(
            positive_scores.mean()
        )


        cos_neg = float(
            negative_scores.mean()
        )


        gap = float(
            cos_pos
            -
            cos_neg
        )


        pooled_std = np.sqrt(

            0.5
            *
            (
                positive_scores.var()
                +
                negative_scores.var()
            )

            +

            1e-12
        )


        d_prime = float(
            gap
            /
            pooled_std
        )


        # ====================================================
        # CHECKPOINT
        # ====================================================

        improved = (
            val_auc
            >
            best_val_auc
            +
            MIN_DELTA
        )


        if improved:

            best_val_auc = (
                val_auc
            )

            best_epoch = (
                epoch
            )

            epochs_without_improvement = 0


            # CPU checkpoint in memory
            best_state = {

                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }


        else:

            epochs_without_improvement += 1


        # ====================================================
        # HISTORY
        # ====================================================

        epoch_result = {

            "epoch":
                epoch,

            "train_loss":
                float(
                    train_metrics["loss"]
                ),

            "val_loss":
                float(
                    val_metrics["loss"]
                ),

            "val_auc":
                val_auc,

            "cos_pos":
                cos_pos,

            "cos_neg":
                cos_neg,

            "gap":
                gap,

            "d_prime":
                d_prime,

            "train_seconds":
                float(
                    train_metrics.get(
                        "seconds",
                        np.nan,
                    )
                ),

            "val_seconds":
                float(
                    val_metrics.get(
                        "seconds",
                        np.nan,
                    )
                ),
        }


        history.append(
            epoch_result
        )


        # ====================================================
        # PRINT
        # ====================================================

        marker = (
            " *BEST*"
            if improved
            else ""
        )


        print(

            f"Epoch "
            f"{epoch:02d}/{MAX_EPOCHS}"

            f" | train loss "
            f"{epoch_result['train_loss']:.5f}"

            f" | val loss "
            f"{epoch_result['val_loss']:.5f}"

            f" | val AUC "
            f"{val_auc:.5f}"

            f" | cos+ "
            f"{cos_pos:.4f}"

            f" | cos- "
            f"{cos_neg:.4f}"

            f" | gap "
            f"{gap:.4f}"

            f" | d' "
            f"{d_prime:.3f}"

            f" | "
            f"{epoch_result['train_seconds']:.1f}s"

            f"{marker}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            epochs_without_improvement
            >=
            EARLY_STOPPING_PATIENCE
        ):

            print()
            print(
                "Early stopping at epoch",
                epoch
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            "No valid checkpoint was created."
        )


    model.load_state_dict(
        best_state
    )


    model.to(
        DEVICE
    )


    history_df = pd.DataFrame(
        history
    )


    print()
    print("=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation ROC-AUC:",
        f"{best_val_auc:.6f}"
    )


    return {
        "model":
            model,

        "history":
            history_df,

        "best_epoch":
            best_epoch,

        "best_val_auc":
            best_val_auc,

        "margin":
            margin,

        "experiment_name":
            experiment_name,

        "test_loader":
            test_loader_exp,
    }

In [54]:
# ============================================================
# CELL 46 — RUN COSINE V3 REFERENCE
# ============================================================

cosine_v3_reference = (
    train_cosine_v3_experiment(

        margin=
            REFERENCE_COSINE_MARGIN,

        experiment_name=
            "cosine_v3_biasfree_margin_0p21875",
    )
)

EXPERIMENT: cosine_v3_biasfree_margin_0p21875
Cosine margin: 0.21875
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.37235 | val loss 0.35455 | val AUC 0.56529 | cos+ 0.7071 | cos- 0.6322 | gap 0.0749 | d' 0.213 | 20.0s *BEST*
Epoch 02/50 | train loss 0.35482 | val loss 0.35181 | val AUC 0.56807 | cos+ 0.7088 | cos- 0.6336 | gap 0.0752 | d' 0.220 | 19.7s *BEST*
Epoch 03/50 | train loss 0.33966 | val loss 0.34480 | val AUC 0.59479 | cos+ 0.7213 | cos- 0.6312 | gap 0.0902 | d' 0.287 | 41.1s *BEST*
Epoch 04/50 | train loss 0.33771 | val loss 0.33705 | val AUC 0.60672 | cos+ 0.7211 | cos- 0.6095 | gap 0.1115 | d' 0.350 | 106.3s *BEST*
Epoch 05/50 | train loss 0.33381 | val loss 0.33568 | val AUC 0.60843 | cos+ 0.6941 | cos- 0.5835 | gap 0.1106 | d' 0.352 | 74.2s *BEST*
Epoch 06/50 | train loss 0.33311 | val loss 0.33501 | val AUC 0.60820 | cos+ 0.6787 | cos- 0.5632 | gap 0.1155 | d' 0.350 | 21.0s
Epoch 07/50 | train l

In [57]:
# ============================================================
# CELL 47 — COSINE V3 MARGIN ABLATION
#
# Reference margin 0.21875 has already been trained.
# Train only the remaining margins.
# ============================================================

COSINE_MARGINS_TO_RUN = [
    0.00,
    0.10,
    0.35,
]


cosine_v3_margin_results = {

    REFERENCE_COSINE_MARGIN:
        cosine_v3_reference
}


for margin in COSINE_MARGINS_TO_RUN:

    margin_tag = (
        str(margin)
        .replace(".", "p")
    )


    experiment_name = (
        f"cosine_v3_biasfree_margin_{margin_tag}"
    )


    print()
    print("#" * 80)
    print(
        "STARTING:",
        experiment_name
    )
    print("#" * 80)
    print()


    result = train_cosine_v3_experiment(

        margin=margin,

        experiment_name=experiment_name,
    )


    cosine_v3_margin_results[
        margin
    ] = result


################################################################################
STARTING: cosine_v3_biasfree_margin_0p0
################################################################################

EXPERIMENT: cosine_v3_biasfree_margin_0p0
Cosine margin: 0.0
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.47701 | val loss 0.45969 | val AUC 0.55786 | cos+ 0.6343 | cos- 0.5576 | gap 0.0766 | d' 0.176 | 30.8s *BEST*
Epoch 02/50 | train loss 0.45237 | val loss 0.45453 | val AUC 0.56675 | cos+ 0.6277 | cos- 0.5403 | gap 0.0874 | d' 0.200 | 50.0s *BEST*
Epoch 03/50 | train loss 0.45012 | val loss 0.43718 | val AUC 0.58997 | cos+ 0.6647 | cos- 0.5408 | gap 0.1238 | d' 0.282 | 115.2s *BEST*
Epoch 04/50 | train loss 0.43881 | val loss 0.43615 | val AUC 0.60054 | cos+ 0.5670 | cos- 0.4254 | gap 0.1416 | d' 0.326 | 114.3s *BEST*
Epoch 05/50 | train loss 0.43088 | val loss 0.43953 | val AUC 0.59787 | cos+ 0.6007 | cos- 

In [58]:
# ============================================================
# CELL 48 — COSINE V3 MARGIN COMPARISON
# ============================================================

margin_summary = []


for margin, result in (
    cosine_v3_margin_results.items()
):

    history = result["history"]

    best_epoch = result["best_epoch"]

    best_row = (
        history[
            history["epoch"]
            ==
            best_epoch
        ]
        .iloc[0]
    )


    margin_summary.append(
        {
            "margin":
                margin,

            "best_epoch":
                best_epoch,

            "val_auc":
                result["best_val_auc"],

            "val_loss":
                float(
                    best_row["val_loss"]
                ),

            "cos_pos":
                float(
                    best_row["cos_pos"]
                ),

            "cos_neg":
                float(
                    best_row["cos_neg"]
                ),

            "gap":
                float(
                    best_row["gap"]
                ),

            "d_prime":
                float(
                    best_row["d_prime"]
                ),
        }
    )


cosine_margin_summary_df = (
    pd.DataFrame(
        margin_summary
    )
    .sort_values(
        "val_auc",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    cosine_margin_summary_df
)


best_cosine_margin = float(
    cosine_margin_summary_df
    .iloc[0]["margin"]
)


print()
print("=" * 70)
print("BEST COSINE V3 CONFIGURATION")
print("=" * 70)

print(
    "Margin:",
    best_cosine_margin
)

print(
    "Validation ROC-AUC:",
    f"{cosine_margin_summary_df.iloc[0]['val_auc']:.6f}"
)

,margin,best_epoch,val_auc,val_loss,cos_pos,cos_neg,gap,d_prime
0,0.35000,37,0.654489,0.258232,0.721291,0.581834,0.139456,0.518848
1,0.21875,42,0.641155,0.318893,0.654827,0.508155,0.146672,0.455734
2,0.10000,38,0.640862,0.362787,0.638347,0.462181,0.176166,0.458475
3,0.00000,44,0.629923,0.405838,0.613592,0.424624,0.188968,0.404092



BEST COSINE V3 CONFIGURATION
Margin: 0.35
Validation ROC-AUC: 0.654489


In [59]:
# ============================================================
# CELL 49 — EXTEND COSINE V3 MARGIN ABLATION
#
# Current best = 0.35, which is at the upper boundary.
# Test two slightly larger margins before touching test data.
# ============================================================

ADDITIONAL_COSINE_MARGINS = [
    0.45,
    0.50,
]


for margin in ADDITIONAL_COSINE_MARGINS:

    margin_tag = (
        str(margin)
        .replace(".", "p")
    )


    experiment_name = (
        f"cosine_v3_biasfree_margin_{margin_tag}"
    )


    print()
    print("#" * 80)
    print(
        "STARTING:",
        experiment_name
    )
    print("#" * 80)
    print()


    result = train_cosine_v3_experiment(

        margin=margin,

        experiment_name=experiment_name,
    )


    cosine_v3_margin_results[
        margin
    ] = result


################################################################################
STARTING: cosine_v3_biasfree_margin_0p45
################################################################################

EXPERIMENT: cosine_v3_biasfree_margin_0p45
Cosine margin: 0.45
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.24419 | val loss 0.24078 | val AUC 0.60072 | cos+ 0.8360 | cos- 0.7707 | gap 0.0653 | d' 0.353 | 27.6s *BEST*
Epoch 02/50 | train loss 0.23596 | val loss 0.23758 | val AUC 0.60988 | cos+ 0.7806 | cos- 0.7060 | gap 0.0747 | d' 0.357 | 20.4s *BEST*
Epoch 03/50 | train loss 0.23349 | val loss 0.23715 | val AUC 0.61426 | cos+ 0.7918 | cos- 0.7121 | gap 0.0796 | d' 0.388 | 20.7s *BEST*
Epoch 04/50 | train loss 0.23372 | val loss 0.23657 | val AUC 0.61766 | cos+ 0.7701 | cos- 0.6885 | gap 0.0816 | d' 0.410 | 46.4s *BEST*
Epoch 05/50 | train loss 0.23317 | val loss 0.23788 | val AUC 0.61315 | cos+ 0.7712 | cos-

In [60]:
# ============================================================
# CELL 50 — FINAL COSINE MARGIN COMPARISON
# ============================================================

margin_summary = []


for margin, result in (
    cosine_v3_margin_results.items()
):

    history = result["history"]

    best_epoch = (
        result["best_epoch"]
    )


    best_row = (
        history[
            history["epoch"]
            ==
            best_epoch
        ]
        .iloc[0]
    )


    margin_summary.append(
        {
            "margin":
                float(margin),

            "best_epoch":
                int(best_epoch),

            "val_auc":
                float(
                    result["best_val_auc"]
                ),

            "val_loss":
                float(
                    best_row["val_loss"]
                ),

            "cos_pos":
                float(
                    best_row["cos_pos"]
                ),

            "cos_neg":
                float(
                    best_row["cos_neg"]
                ),

            "gap":
                float(
                    best_row["gap"]
                ),

            "d_prime":
                float(
                    best_row["d_prime"]
                ),
        }
    )


cosine_margin_final_df = (
    pd.DataFrame(
        margin_summary
    )
    .sort_values(
        "val_auc",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    cosine_margin_final_df
)


best_cosine_margin = float(
    cosine_margin_final_df
    .iloc[0]["margin"]
)


best_cosine_val_auc = float(
    cosine_margin_final_df
    .iloc[0]["val_auc"]
)


print()
print("=" * 70)
print("FINAL BEST COSINE CONFIGURATION")
print("=" * 70)

print(
    "Margin:",
    best_cosine_margin
)

print(
    "Validation ROC-AUC:",
    f"{best_cosine_val_auc:.6f}"
)

,margin,best_epoch,val_auc,val_loss,cos_pos,cos_neg,gap,d_prime
0,0.50000,22,0.654550,0.198905,0.783665,0.678513,0.105152,0.527464
1,0.35000,37,0.654489,0.258232,0.721291,0.581834,0.139456,0.518848
2,0.45000,20,0.651842,0.222783,0.750260,0.638935,0.111324,0.507868
3,0.21875,42,0.641155,0.318893,0.654827,0.508155,0.146672,0.455734
4,0.10000,38,0.640862,0.362787,0.638347,0.462181,0.176166,0.458475
5,0.00000,44,0.629923,0.405838,0.613592,0.424624,0.188968,0.404092



FINAL BEST COSINE CONFIGURATION
Margin: 0.5
Validation ROC-AUC: 0.654550


In [61]:
# ============================================================
# CELL 51 — OVERNIGHT COSINE MARGIN EXTENSION
#
# Only two additional points.
# Do not touch the test set.
# ============================================================

OVERNIGHT_COSINE_MARGINS = [
    0.60,
    0.70,
]


for margin in OVERNIGHT_COSINE_MARGINS:

    margin_tag = (
        str(margin)
        .replace(".", "p")
    )

    experiment_name = (
        f"cosine_v3_biasfree_margin_{margin_tag}"
    )

    print()
    print("#" * 80)
    print(
        "STARTING:",
        experiment_name
    )
    print("#" * 80)
    print()

    result = train_cosine_v3_experiment(
        margin=margin,
        experiment_name=experiment_name,
    )

    cosine_v3_margin_results[
        margin
    ] = result


################################################################################
STARTING: cosine_v3_biasfree_margin_0p6
################################################################################

EXPERIMENT: cosine_v3_biasfree_margin_0p6
Cosine margin: 0.6
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.18004 | val loss 0.18495 | val AUC 0.55903 | cos+ 0.9139 | cos- 0.8878 | gap 0.0260 | d' 0.182 | 20.9s *BEST*
Epoch 02/50 | train loss 0.17278 | val loss 0.17350 | val AUC 0.61061 | cos+ 0.8605 | cos- 0.8088 | gap 0.0516 | d' 0.359 | 23.3s *BEST*
Epoch 03/50 | train loss 0.16990 | val loss 0.17267 | val AUC 0.61749 | cos+ 0.8540 | cos- 0.8003 | gap 0.0537 | d' 0.404 | 105.4s *BEST*
Epoch 04/50 | train loss 0.17111 | val loss 0.17328 | val AUC 0.61853 | cos+ 0.8131 | cos- 0.7471 | gap 0.0660 | d' 0.404 | 89.4s *BEST*
Epoch 05/50 | train loss 0.16921 | val loss 0.17036 | val AUC 0.61612 | cos+ 0.8345 | cos- 0

In [63]:
# ============================================================
# CELL 53 — FINAL BOUNDARY CHECK: MARGIN 0.80
# ============================================================

margin = 0.80

result_080 = train_cosine_v3_experiment(
    margin=margin,
    experiment_name="cosine_v3_biasfree_margin_0p80",
)

cosine_v3_margin_results[
    margin
] = result_080

EXPERIMENT: cosine_v3_biasfree_margin_0p80
Cosine margin: 0.8
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.09127 | val loss 0.08832 | val AUC 0.60634 | cos+ 0.9578 | cos- 0.9351 | gap 0.0227 | d' 0.359 | 21.6s *BEST*
Epoch 02/50 | train loss 0.08714 | val loss 0.08599 | val AUC 0.61730 | cos+ 0.9299 | cos- 0.9019 | gap 0.0280 | d' 0.403 | 22.5s *BEST*
Epoch 03/50 | train loss 0.08550 | val loss 0.08724 | val AUC 0.61276 | cos+ 0.9374 | cos- 0.9128 | gap 0.0247 | d' 0.386 | 22.5s
Epoch 04/50 | train loss 0.08565 | val loss 0.08604 | val AUC 0.61935 | cos+ 0.9144 | cos- 0.8836 | gap 0.0308 | d' 0.413 | 22.6s *BEST*
Epoch 05/50 | train loss 0.08396 | val loss 0.08448 | val AUC 0.62948 | cos+ 0.9173 | cos- 0.8825 | gap 0.0348 | d' 0.451 | 20.2s *BEST*
Epoch 06/50 | train loss 0.08361 | val loss 0.08320 | val AUC 0.63745 | cos+ 0.9206 | cos- 0.8860 | gap 0.0346 | d' 0.475 | 69.5s *BEST*
Epoch 07/50 | train loss 0.08

In [64]:
# ============================================================
# CELL 52 — OVERNIGHT FINAL SUMMARY
# ============================================================

overnight_summary = []


for margin, result in cosine_v3_margin_results.items():

    history = result["history"]

    best_epoch = result["best_epoch"]

    best_row = (
        history[
            history["epoch"] == best_epoch
        ]
        .iloc[0]
    )

    overnight_summary.append(
        {
            "margin":
                float(margin),

            "best_epoch":
                int(best_epoch),

            "val_auc":
                float(
                    result["best_val_auc"]
                ),

            "val_loss":
                float(
                    best_row["val_loss"]
                ),

            "cos_pos":
                float(
                    best_row["cos_pos"]
                ),

            "cos_neg":
                float(
                    best_row["cos_neg"]
                ),

            "gap":
                float(
                    best_row["gap"]
                ),

            "d_prime":
                float(
                    best_row["d_prime"]
                ),
        }
    )


overnight_summary_df = (
    pd.DataFrame(
        overnight_summary
    )
    .sort_values(
        "val_auc",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    overnight_summary_df
)


print()
print("=" * 70)
print("BEST COSINE CONFIGURATION AFTER OVERNIGHT RUNS")
print("=" * 70)

print(
    "Margin:",
    overnight_summary_df.iloc[0]["margin"]
)

print(
    "Validation ROC-AUC:",
    f"{overnight_summary_df.iloc[0]['val_auc']:.6f}"
)

print(
    "d-prime:",
    f"{overnight_summary_df.iloc[0]['d_prime']:.6f}"
)

,margin,best_epoch,val_auc,val_loss,cos_pos,cos_neg,gap,d_prime
0,0.80000,46,0.659911,0.079082,0.912439,0.868127,0.044311,0.533434
1,0.70000,46,0.657201,0.118772,0.867044,0.801033,0.066011,0.526738
2,0.50000,22,0.654550,0.198905,0.783665,0.678513,0.105152,0.527464
3,0.35000,37,0.654489,0.258232,0.721291,0.581834,0.139456,0.518848
4,0.60000,18,0.653643,0.160698,0.825663,0.744777,0.080886,0.513868
5,0.45000,20,0.651842,0.222783,0.750260,0.638935,0.111324,0.507868
6,0.21875,42,0.641155,0.318893,0.654827,0.508155,0.146672,0.455734
7,0.10000,38,0.640862,0.362787,0.638347,0.462181,0.176166,0.458475
8,0.00000,44,0.629923,0.405838,0.613592,0.424624,0.188968,0.404092



BEST COSINE CONFIGURATION AFTER OVERNIGHT RUNS
Margin: 0.8
Validation ROC-AUC: 0.659911
d-prime: 0.533434


In [70]:
# ============================================================
# SAVE BEST COSINE V3 MODEL BEFORE KERNEL RESTART
# ============================================================

from pathlib import Path
import torch
import json

SAVE_DIR = Path("../artifacts/cosine_v3")
SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

best_cosine_result = cosine_v3_margin_results[
    0.80
]

best_cosine_model = best_cosine_result[
    "model"
]


# ------------------------------------------------------------
# SAVE MODEL CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = (
    SAVE_DIR
    /
    "cosine_v3_margin_0p80_best.pt"
)

torch.save(
    {
        "model_state_dict":
            best_cosine_model.state_dict(),

        "embedding_dim":
            EMBEDDING_DIM,

        "margin":
            0.80,

        "best_epoch":
            best_cosine_result["best_epoch"],

        "best_val_auc":
            best_cosine_result["best_val_auc"],

        "architecture":
            "SiameseNetworkCosineV3",

        "conv_bias":
            False,
    },
    checkpoint_path,
)


# ------------------------------------------------------------
# SAVE TRAINING HISTORY
# ------------------------------------------------------------

history_path = (
    SAVE_DIR
    /
    "cosine_v3_margin_0p80_history.tsv"
)

best_cosine_result[
    "history"
].to_csv(
    history_path,
    sep="\t",
    index=False,
)


print("=" * 70)
print("COSINE V3 SAVED")
print("=" * 70)

print(
    "Checkpoint:",
    checkpoint_path
)

print(
    "History:",
    history_path
)

print(
    "Best epoch:",
    best_cosine_result["best_epoch"]
)

print(
    "Best Val ROC-AUC:",
    best_cosine_result["best_val_auc"]
)

COSINE V3 SAVED
Checkpoint: ..\artifacts\cosine_v3\cosine_v3_margin_0p80_best.pt
History: ..\artifacts\cosine_v3\cosine_v3_margin_0p80_history.tsv
Best epoch: 46
Best Val ROC-AUC: 0.6599109638794127


In [71]:
# ============================================================
# SAVE COSINE MARGIN ABLATION SUMMARY
# ============================================================

margin_ablation_path = (
    SAVE_DIR
    /
    "cosine_v3_margin_ablation.tsv"
)

cosine_margin_final_df.to_csv(
    margin_ablation_path,
    sep="\t",
    index=False,
)

print(
    "Margin ablation saved:",
    margin_ablation_path
)

Margin ablation saved: ..\artifacts\cosine_v3\cosine_v3_margin_ablation.tsv


In [72]:
checkpoint = torch.load(
    "../artifacts/cosine_v3/cosine_v3_margin_0p80_best.pt",
    map_location=DEVICE,
)

best_cosine_model = SiameseNetworkCosineV3(
    embedding_dim=checkpoint["embedding_dim"]
).to(DEVICE)

best_cosine_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_cosine_model.eval()

print(
    "Loaded margin:",
    checkpoint["margin"]
)

print(
    "Best epoch:",
    checkpoint["best_epoch"]
)

print(
    "Best Val AUC:",
    checkpoint["best_val_auc"]
)

Loaded margin: 0.8
Best epoch: 46
Best Val AUC: 0.6599109638794127
